# Bangladeshi Meme Classification with Qwen-VL + LoRA

**Task.** Single-label classification of Bangladeshi memes into five categories:
`Political`, `Religious`, `Sports`, `Educational`, `Neutral` (or `Harmless`).

**Method.** Parameter-efficient supervised fine-tuning (LoRA) of a Qwen-VL instruction-tuned
vision-language model (e.g. `Qwen/Qwen2.5-VL-3B-Instruct` or `Qwen/Qwen2-VL-2B-Instruct`).
The model is trained *generatively*: given the meme image and a fixed classification instruction
it emits the class name. At inference time we score all candidate class strings under the model
and take the arg-max. That yields a calibrated probability distribution per class.

**Pipeline.**
dataset discovery (CSV / folder-based) -> quality analysis -> duplicate & near-duplicate detection
-> leakage-safe stratified split -> augmentation -> Qwen-VL + LoRA fine-tuning -> validation-based
model selection -> single final test evaluation -> confusion matrix -> embeddings -> t-SNE/UMAP
-> class-centroid similarity -> misclassification analysis -> artifact export.

**Dataset Support.**
Automatically discovers either:
1. **CSV-annotated datasets**: e.g. `labels.csv` with `Image_name,Label` + images in `Train/` or flat directory.
2. **Folder-annotated datasets**: e.g. `<Class>/*.jpg` or `<split>/<Class>/*.jpg`.


---
## 01. Research Configuration

Everything tunable lives in one dictionary, so a reviewer never has to hunt for a magic number
further down.

On model size: Kaggle's T4 (16 GB) and P100 (16 GB) accelerators are the realistic target.
`Qwen/Qwen2.5-VL-3B-Instruct` or `Qwen/Qwen2-VL-2B-Instruct` trains comfortably with LoRA/QLoRA,
bf16/fp16 and capped image resolution. Change `model_id` if you have more VRAM.

In [ ]:
import os, sys, json, math, random, hashlib, warnings, time
from pathlib import Path
from collections import defaultdict
import pandas as pd
import numpy as np

warnings.filterwarnings("ignore")

CONFIG = {
    # ---------------- experiment identity ----------------
    "experiment_name": "qwen_vl_lora_bd_meme_5class",
    "seed": 42,

    # ---------------- classes (FIXED ORDER, used by every table and figure) ----------------
    "classes": ["Political", "Religious", "Sports", "Educational", "Neutral"],

    # ---------------- dataset discovery ----------------
    "search_roots": ["/kaggle/input", "/kaggle/working/data", "/kaggle/working", "./train_image", "./data", "."],
    "image_extensions": [".jpg", ".jpeg", ".png", ".webp", ".bmp", ".gif", ".tif", ".tiff"],
    # label/folder aliases -> canonical class name
    "class_aliases": {
        "political": "Political", "politics": "Political", "politic": "Political",
        "politicalmemes": "Political", "politicalmeme": "Political",
        "religious": "Religious", "religion": "Religious", "religiousmemes": "Religious",
        "sports": "Sports", "sport": "Sports", "sportsmemes": "Sports", "sportsmeme": "Sports",
        "educational": "Educational", "education": "Educational", "educationalmemes": "Educational",
        "harmless": "Neutral", "neutral": "Neutral", "harmlessmemes": "Neutral",
        "nonharmful": "Neutral", "nonharmless": "Neutral",
    },

    # ---------------- duplicate detection ----------------
    "phash_size": 8,             # 8x8 DCT low-frequency block -> 64-bit perceptual hash
    "near_dup_hamming_max": 5,   # <= 5 differing bits out of 64 counts as a near duplicate
    "run_near_dup": True,

    # ---------------- split ----------------
    "split": {"train": 0.70, "val": 0.15, "test": 0.15},

    # ---------------- model ----------------
    "model_id": "Qwen/Qwen2.5-VL-3B-Instruct",
    "attn_implementation": "sdpa",   # "flash_attention_2" only when flash-attn wheel is installed

    # ---------------- image tokenisation budget ----------------
    "min_pixels": 64 * 28 * 28,
    "max_pixels": 256 * 28 * 28,

    # ---------------- LoRA ----------------
    "lora": {
        "r": 16, "alpha": 32, "dropout": 0.05, "bias": "none",
        "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj",
                           "gate_proj", "up_proj", "down_proj"],
        "adapt_vision_tower": False,
    },

    # ---------------- training ----------------
    "train": {
        "epochs": 5,
        "per_device_batch_size": 1,
        "grad_accum_steps": 8,
        "lr": 1e-4,
        "weight_decay": 0.0,
        "warmup_ratio": 0.05,
        "max_grad_norm": 1.0,
        "gradient_checkpointing": True,
        "early_stopping_patience": 2,
        "eval_batch_images": 2,
        "log_every_steps": 10,
        "max_train_samples": None,
        "max_eval_samples": None,
    },

    # ---------------- augmentation ----------------
    "aug": {
        "hflip_p": 0.0,
        "rotation_deg": 5.0, "rotation_p": 0.30,
        "brightness": 0.12, "contrast": 0.12, "saturation": 0.08, "jitter_p": 0.30,
        "random_resized_crop_p": 0.25, "crop_scale": (0.88, 1.0),
    },

    # ---------------- output ----------------
    "out_dir": "/kaggle/working/meme_classification_results",
}

CLASSES = CONFIG["classes"]
NUM_CLASSES = len(CLASSES)
CLASS_TO_ID = {c: i for i, c in enumerate(CLASSES)}
ID_TO_CLASS = {i: c for c, i in CLASS_TO_ID.items()}

# Fixed colour per class
CLASS_COLORS = {"Political": "#4C72B0", "Religious": "#DD8452", "Sports": "#55A868",
                "Educational": "#C44E52", "Neutral": "#8172B3", "Harmless": "#8172B3"}

OUT = Path(CONFIG["out_dir"])
for sub in ["model", "splits", "metrics", "plots", "predictions", "embeddings", "logs"]:
    (OUT / sub).mkdir(parents=True, exist_ok=True)

print("Experiment :", CONFIG["experiment_name"])
print("Classes    :", CLASSES)
print("Output dir :", OUT)

### 01b. Reproducibility

Seeds for Python, NumPy and PyTorch, plus deterministic cuDNN. Full bit-wise determinism is still
not guaranteed for fused attention kernels on GPU, so we persist the seed and the split index
(Section 07) rather than claiming the weights are exactly reproducible.

In [ ]:
import numpy as np
import torch

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SEED = CONFIG["seed"]
set_seed(SEED)
print("Global seed set to", SEED)

---
## 02. Environment & GPU Check

We inspect what this Kaggle session provides before touching the model. Qwen2.5-VL / Qwen-VL needs
`transformers >= 4.49.0` (or `transformers >= 4.45.0` for Qwen2-VL). The installer cell runs only when
a package is missing or older than required, and it avoids unnecessary re-installations.

In [ ]:
import importlib, subprocess, platform

def pkg_version(name):
    try:
        return importlib.import_module(name).__version__
    except Exception:
        try:
            from importlib.metadata import version
            return version(name)
        except Exception:
            return None

def parse_ver(v):
    if v is None:
        return (0, 0, 0)
    parts = []
    for p in str(v).split(".")[:3]:
        digits = "".join(ch for ch in p if ch.isdigit())
        parts.append(int(digits) if digits else 0)
    while len(parts) < 3:
        parts.append(0)
    return tuple(parts)

ENV_INFO = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "torch": pkg_version("torch"),
    "torchvision": pkg_version("torchvision"),
    "transformers": pkg_version("transformers"),
    "peft": pkg_version("peft"),
    "accelerate": pkg_version("accelerate"),
    "bitsandbytes": pkg_version("bitsandbytes"),
    "sklearn": pkg_version("sklearn"),
    "pillow": pkg_version("PIL"),
    "umap_learn": pkg_version("umap"),
    "cuda_available": torch.cuda.is_available(),
    "cuda_version": torch.version.cuda,
    "n_gpu": torch.cuda.device_count(),
    "seed": SEED,
}
for k, v in ENV_INFO.items():
    print(f"{k:<16}: {v}")

GPU_INFO = []
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        GPU_INFO.append({"index": i, "name": p.name,
                         "total_memory_gib": round(p.total_memory / 1024**3, 2),
                         "capability": f"{p.major}.{p.minor}"})
        print(f"  GPU {i}: {p.name} | {p.total_memory/1024**3:.1f} GiB | sm_{p.major}{p.minor}")
else:
    print("  !! No CUDA device visible. Enable a GPU accelerator in the Kaggle sidebar.")
ENV_INFO["gpus"] = GPU_INFO

INTERNET = False
try:
    import socket
    socket.create_connection(("huggingface.co", 443), timeout=5).close()
    INTERNET = True
except Exception:
    pass
ENV_INFO["internet"] = INTERNET
print("Internet reachable:", INTERNET)

In [ ]:
# ---- Conditional dependency upgrade -------------------------------------------------------
REQUIRED = {
    "transformers": (4, 49, 0),
    "peft":         (0, 13, 0),
    "accelerate":   (0, 34, 0),
}
missing = {n: (pkg_version(n), req) for n, req in REQUIRED.items()
           if parse_ver(pkg_version(n)) < req}

if missing:
    print("Upgrade needed:")
    for n, (have, req) in missing.items():
        print(f"  {n}: installed {have}, required >= {'.'.join(map(str, req))}")
    if not INTERNET:
        print("Warning: Internet is off. If import fails, please enable Internet in Kaggle notebook settings.")
    else:
        specs = [f"{n}>={'.'.join(map(str, req))}" for n, (_, req) in missing.items()]
        print("Installing:", specs)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *specs], check=False)
        print("\n*** If this is the first run after upgrading, restart kernel and run from the top. ***")
else:
    print("All core dependencies meet the required versions. Nothing installed.")

# Optional extra: umap-learn for feature visualization
if pkg_version("umap") is None and INTERNET:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "umap-learn"], check=False)
print("umap-learn:", pkg_version("umap"))

---
## 03. Dataset Discovery

The dataset discovery is completely dynamic and handles multiple layouts automatically:

1. **CSV-Annotated Dataset** (e.g. `labels.csv` or `train.csv`):
   - Auto-detects the image column (`Image_name`, `filename`, `image`, etc.) and label column (`Label`, `category`, `class`, etc.).
   - Locates image files in subfolders (e.g. `Train/`, `images/`, `train_image/Train/`, or flat folder).
2. **Folder-Annotated Dataset**:
   - Walks candidate roots for class folders (e.g. `root/<Class>/*.jpg` or `root/<split>/<Class>/*.jpg`).

If class names are formatted differently (e.g. `Neutral` vs `Harmless`), aliases automatically map them to the 5 canonical classes.

In [ ]:
import pandas as pd
IMG_EXT = set(CONFIG["image_extensions"])

def normalise_str(name: str) -> str:
    """Lowercase and strip separators so 'Political_Memes' and 'political memes' collide."""
    s = str(name).strip().lower()
    for ch in [" ", "_", "-", ".", "(", ")"]:
        s = s.replace(ch, "")
    return s

def canonical_class(label_name: str):
    """Map a label or folder name to one of the five canonical classes, or None."""
    if label_name is None or pd.isna(label_name):
        return None
    key = normalise_str(label_name)
    if key in CONFIG["class_aliases"]:
        return CONFIG["class_aliases"][key]
    for c in CLASSES:
        if key == normalise_str(c):
            return c
    for c in CLASSES:
        if normalise_str(c) in key:
            return c
    return None

def count_images_in(d: Path) -> int:
    n = 0
    try:
        for e in os.scandir(d):
            if e.is_file() and Path(e.name).suffix.lower() in IMG_EXT:
                n += 1
    except (PermissionError, OSError):
        pass
    return n

def index_all_images_on_disk(roots, max_depth=4):
    """Index all image files across roots: {filename.lower(): full_path}."""
    img_map = {}
    for r in roots:
        p = Path(r)
        if not p.exists():
            continue
        for dirpath, dirnames, filenames in os.walk(p):
            depth = len(Path(dirpath).relative_to(p).parts)
            if depth > max_depth or ".git" in dirpath or ".ipynb_checkpoints" in dirpath:
                dirnames[:] = []
                continue
            for fn in filenames:
                if Path(fn).suffix.lower() in IMG_EXT:
                    img_map[fn.lower()] = str((Path(dirpath) / fn).resolve())
    return img_map

def discover_from_csv(roots, img_map, max_depth=4):
    """Try discovering dataset from CSV metadata (e.g. labels.csv)."""
    for r in roots:
        p = Path(r)
        if not p.exists():
            continue
        for dirpath, dirnames, filenames in os.walk(p):
            depth = len(Path(dirpath).relative_to(p).parts)
            if depth > max_depth or ".git" in dirpath or ".ipynb_checkpoints" in dirpath:
                dirnames[:] = []
                continue
            for fn in filenames:
                if fn.lower().endswith(".csv") and not fn.startswith("submission"):
                    csv_path = Path(dirpath) / fn
                    try:
                        df = pd.read_csv(csv_path)
                    except Exception:
                        continue
                    img_col, lbl_col = None, None
                    for col in df.columns:
                        c_norm = normalise_str(col)
                        if c_norm in ["imagename", "image", "filename", "file", "id", "img", "imagepath", "filepath"]:
                            img_col = col
                            break
                    for col in df.columns:
                        c_norm = normalise_str(col)
                        if c_norm in ["label", "category", "class", "target", "intent", "satire"]:
                            lbl_col = col
                            break
                    if img_col and lbl_col:
                        records = []
                        for _, row in df.iterrows():
                            img_name = str(row[img_col]).strip()
                            canon_lbl = canonical_class(row[lbl_col])
                            if not canon_lbl:
                                continue
                            img_file = None
                            cand1 = Path(dirpath) / img_name
                            cand2 = Path(dirpath) / "Train" / img_name
                            cand3 = Path(dirpath) / "train" / img_name
                            cand4 = Path(dirpath) / "images" / img_name
                            if cand1.exists(): img_file = str(cand1.resolve())
                            elif cand2.exists(): img_file = str(cand2.resolve())
                            elif cand3.exists(): img_file = str(cand3.resolve())
                            elif cand4.exists(): img_file = str(cand4.resolve())
                            elif img_name.lower() in img_map:
                                img_file = img_map[img_name.lower()]
                            if img_file and Path(img_file).exists():
                                records.append({
                                    "filepath": img_file,
                                    "filename": Path(img_file).name,
                                    "class": canon_lbl,
                                    "label_id": CLASS_TO_ID[canon_lbl],
                                    "source_dir": str(Path(img_file).parent),
                                    "extension": Path(img_file).suffix.lower()
                                })
                        if len(records) > 0:
                            print(f"[CSV Match] Discovered {len(records)} images via: {csv_path}")
                            return records
    return None

def discover_from_folders(roots, max_depth=4):
    """Fallback: discover from class-named directories."""
    found = defaultdict(list)
    for r in roots:
        p = Path(r)
        if not p.exists(): continue
        for dirpath, dirnames, filenames in os.walk(p):
            depth = len(Path(dirpath).relative_to(p).parts)
            if depth > max_depth or ".git" in dirpath:
                dirnames[:] = []
                continue
            cls = canonical_class(Path(dirpath).name)
            if cls is not None and count_images_in(Path(dirpath)) > 0:
                found[cls].append(Path(dirpath))
    return found

print("Scanning candidate roots for dataset ...\n")
ALL_IMAGES_MAP = index_all_images_on_disk(CONFIG["search_roots"])
print(f"Total image files indexed on disk: {len(ALL_IMAGES_MAP)}")

RECORDS = discover_from_csv(CONFIG["search_roots"], ALL_IMAGES_MAP)
if not RECORDS:
    print("No matching CSV metadata found. Falling back to folder-based discovery...")
    CLASS_DIRS = discover_from_folders(CONFIG["search_roots"])
    RECORDS = []
    seen_paths = set()
    for cls, dirs in CLASS_DIRS.items():
        for d in dirs:
            for e in sorted(os.scandir(d), key=lambda x: x.name):
                if not e.is_file(): continue
                p = Path(e.path)
                if p.suffix.lower() not in IMG_EXT: continue
                rp = str(p.resolve())
                if rp in seen_paths: continue
                seen_paths.add(rp)
                RECORDS.append({"filepath": rp, "filename": p.name, "class": cls,
                                "label_id": CLASS_TO_ID[cls], "source_dir": str(d),
                                "extension": p.suffix.lower()})

if not RECORDS:
    print("Directory tree of /kaggle/input:")
    base = Path("/kaggle/input")
    if base.exists():
        for d1 in sorted(base.iterdir()):
            print(" ", d1.name)
            if d1.is_dir():
                for d2 in sorted(d1.iterdir())[:20]:
                    print("    ", d2.name)
    raise SystemExit("Stopping: could not locate dataset files under search roots. "
                     "Please ensure the dataset is attached in Kaggle.")

print(f"\nSuccessfully discovered {len(RECORDS)} annotated image records.")

In [ ]:
# ---- Build the raw file dataframe ---------------------------------------------------------
import pandas as pd
raw_df = pd.DataFrame(RECORDS)
print(f"Raw image files found: {len(raw_df)}")
print("\nClass counts in raw dataset:")
print(raw_df["class"].value_counts().reindex(CLASSES).to_string())
print("\nExtensions:")
print(raw_df["extension"].value_counts().to_string())

---
## 04. Dataset Quality Analysis

Each file is opened once. We record width, height, mode, aspect ratio and size on disk, and we flag
anything a reader of the thesis would want disclosed: unreadable files, truncated files, grayscale
or RGBA images, and extreme dimensions.

Nothing is deleted silently. Files that cannot be decoded at all are excluded from training (they
are not usable input) and are listed explicitly, with the count reported in the final summary.

In [ ]:
from PIL import Image, ImageFile, UnidentifiedImageError
Image.MAX_IMAGE_PIXELS = None          # we check size ourselves rather than tripping the bomb guard
ImageFile.LOAD_TRUNCATED_IMAGES = False  # we WANT truncated files to raise so they get flagged

SMALL_SIDE_THRESHOLD = 64        # below this a meme is unreadable even to a human
LARGE_SIDE_THRESHOLD = 4000      # above this, decoding cost dominates

def inspect_image(path: str) -> dict:
    info = {"readable": False, "width": None, "height": None, "mode": None,
            "format": None, "n_frames": 1, "file_size": None, "error": None}
    try:
        info["file_size"] = os.path.getsize(path)
    except OSError as ex:
        info["error"] = f"stat failed: {ex}"
        return info
    try:
        with Image.open(path) as im:
            im.verify()                        # catches truncation / CRC problems
        with Image.open(path) as im:
            info.update(width=im.width, height=im.height, mode=im.mode,
                        format=im.format, n_frames=getattr(im, "n_frames", 1))
            im.load()                          # force a real decode
        info["readable"] = True
    except (UnidentifiedImageError, OSError, SyntaxError, ValueError) as ex:
        info["error"] = f"{type(ex).__name__}: {ex}"
    return info

print(f"Inspecting {len(raw_df)} files (one decode each) ...")
t0 = time.time()
insp = []
for i, p in enumerate(raw_df["filepath"].tolist()):
    insp.append(inspect_image(p))
    if (i + 1) % 500 == 0:
        print(f"  {i+1}/{len(raw_df)}  ({time.time()-t0:.0f}s)")
insp_df = pd.DataFrame(insp)
index_df = pd.concat([raw_df.reset_index(drop=True), insp_df], axis=1)
print(f"Done in {time.time()-t0:.0f}s")

index_df["aspect_ratio"] = index_df["width"] / index_df["height"]
index_df["megapixels"] = (index_df["width"] * index_df["height"]) / 1e6
index_df["file_size_kb"] = index_df["file_size"] / 1024.0
index_df["is_grayscale"] = index_df["mode"].isin(["L", "LA", "1", "I", "F"])
index_df["has_alpha"] = index_df["mode"].isin(["RGBA", "LA", "PA"])
index_df["is_animated"] = index_df["n_frames"].fillna(1) > 1
index_df["is_tiny"] = index_df[["width", "height"]].min(axis=1) < SMALL_SIDE_THRESHOLD
index_df["is_huge"] = index_df[["width", "height"]].max(axis=1) > LARGE_SIDE_THRESHOLD

In [ ]:
# ---- Quality report ----------------------------------------------------------------------
n_total = len(index_df)
bad = index_df[~index_df["readable"]]
ok = index_df[index_df["readable"]].copy()

print("=" * 78)
print("DATASET QUALITY REPORT")
print("=" * 78)
print(f"Files discovered      : {n_total}")
print(f"Decoded successfully  : {len(ok)}")
print(f"Unreadable / corrupt  : {len(bad)}")
if len(bad):
    print("\nUnreadable files (excluded from training, retained in the index):")
    for _, r in bad.iterrows():
        print(f"  [{r['class']}] {r['filepath']}  ->  {r['error']}")

flag_cols = ["is_grayscale", "has_alpha", "is_animated", "is_tiny", "is_huge"]
print("\nFlagged-but-usable files:")
for c in flag_cols:
    sub = ok[ok[c]]
    print(f"  {c:<14}: {len(sub)}")
    if 0 < len(sub) <= 12:
        for _, r in sub.iterrows():
            print(f"        [{r['class']}] {r['filename']} ({r['mode']}, {r['width']}x{r['height']})")

print("\nDimension statistics over decodable images:")
dim_stats = ok[["width", "height", "aspect_ratio", "file_size_kb", "megapixels"]].describe(
    percentiles=[0.25, 0.5, 0.75]).T[["count", "mean", "std", "min", "25%", "50%", "75%", "max"]]
print(dim_stats.round(2).to_string())

print("\nColour modes present:", dict(ok["mode"].value_counts()))
print("File formats present:", dict(ok["format"].value_counts()))

In [ ]:
# ---- Class distribution table -------------------------------------------------------------
dist = (ok.groupby("class").size().reindex(CLASSES).fillna(0).astype(int)
        .rename("n_images").to_frame())
dist["percentage"] = (dist["n_images"] / dist["n_images"].sum() * 100).round(2)
dist["mean_width"] = ok.groupby("class")["width"].mean().reindex(CLASSES).round(1)
dist["mean_height"] = ok.groupby("class")["height"].mean().reindex(CLASSES).round(1)
dist["median_ar"] = ok.groupby("class")["aspect_ratio"].median().reindex(CLASSES).round(3)
dist["mean_kb"] = ok.groupby("class")["file_size_kb"].mean().reindex(CLASSES).round(1)

print("Class | Number of Images | Percentage")
print(dist.to_string())

imb = dist["n_images"].max() / max(dist["n_images"].min(), 1)
print(f"\nImbalance ratio (largest / smallest class): {imb:.2f}")
if imb > 3:
    print("  -> Strong imbalance. Macro-F1 is the primary metric; accuracy alone would mislead.")
elif imb > 1.5:
    print("  -> Moderate imbalance. Macro-F1 is reported alongside accuracy throughout.")
else:
    print("  -> Classes are roughly balanced.")

CLEAN = ok.reset_index(drop=True)

---
## 05. Duplicate & Leakage Detection

Two passes.

**Exact duplicates** use the SHA-256 of the raw file bytes. Byte-identical files are unambiguous
duplicates, including the case where the same meme sits in two different class folders, which is a
labelling conflict worth reporting.

**Near duplicates** use a 64-bit perceptual hash (pHash): grayscale, resize to 32x32, 2-D DCT, keep
the top-left 8x8 low-frequency block excluding the DC term, threshold against the median. Two
images are near-duplicates when their hashes differ in at most `near_dup_hamming_max` bits. The
comparison is accelerated by bucketing on hash halves before the pairwise check, so we avoid the
full O(n^2) sweep on larger corpora but still catch everything within the threshold.

Duplicates are then grouped with a union-find into connected components. **The split in Section 07
assigns whole components**, which is what prevents a meme from appearing in both train and test.

In [ ]:
import scipy.fftpack as fftpack

def sha256_of_file(path: str, chunk: int = 1 << 20) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for blk in iter(lambda: f.read(chunk), b""):
            h.update(blk)
    return h.hexdigest()

def phash_bits(path: str, hash_size: int = 8, highfreq_factor: int = 4):
    """64-bit DCT perceptual hash as a numpy bool array, or None when unreadable."""
    img_size = hash_size * highfreq_factor
    try:
        with Image.open(path) as im:
            im = im.convert("L").resize((img_size, img_size), Image.Resampling.LANCZOS)
            arr = np.asarray(im, dtype=np.float64)
    except Exception:
        return None
    dct = fftpack.dct(fftpack.dct(arr, axis=0, norm="ortho"), axis=1, norm="ortho")
    low = dct[:hash_size, :hash_size].flatten()
    med = np.median(low[1:])          # drop the DC term before taking the median
    return low > med

def bits_to_hex(bits) -> str:
    return format(int("".join("1" if b else "0" for b in bits), 2), "016x")

print("Hashing ...")
t0 = time.time()
sha_list, ph_bits, ph_hex = [], [], []
for i, p in enumerate(CLEAN["filepath"].tolist()):
    sha_list.append(sha256_of_file(p))
    b = phash_bits(p, CONFIG["phash_size"]) if CONFIG["run_near_dup"] else None
    ph_bits.append(b)
    ph_hex.append(bits_to_hex(b) if b is not None else None)
    if (i + 1) % 500 == 0:
        print(f"  {i+1}/{len(CLEAN)}  ({time.time()-t0:.0f}s)")
CLEAN["sha256"] = sha_list
CLEAN["phash"] = ph_hex
print(f"Hashing done in {time.time()-t0:.0f}s")

In [ ]:
# ---- Union-find over duplicate relations --------------------------------------------------
parent = list(range(len(CLEAN)))

def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[max(ra, rb)] = min(ra, rb)

# --- exact duplicates ---
exact_groups, exact_pairs = [], 0
for sha, idxs in CLEAN.groupby("sha256").groups.items():
    idxs = list(idxs)
    if len(idxs) > 1:
        exact_groups.append(idxs)
        for j in idxs[1:]:
            union(idxs[0], j)
            exact_pairs += 1

n_exact_extra = sum(len(g) - 1 for g in exact_groups)
print(f"Exact-duplicate groups            : {len(exact_groups)}")
print(f"Redundant exact-duplicate copies  : {n_exact_extra}")

cross_label_exact = []
for g in exact_groups:
    labs = sorted(set(CLEAN.loc[g, "class"]))
    if len(labs) > 1:
        cross_label_exact.append((labs, [CLEAN.loc[i, "filename"] for i in g]))
print(f"Exact duplicates spanning >1 class: {len(cross_label_exact)}")
for labs, files in cross_label_exact[:20]:
    print(f"   {labs} <- {files}")
if cross_label_exact:
    print("   NOTE: identical bytes under two labels is a label conflict, not just redundancy. "
          "These are reported, kept in one split, and flagged in the final summary.")

In [ ]:
# ---- Near duplicates via banded pHash buckets ----------------------------------------------
near_pairs = []
if CONFIG["run_near_dup"]:
    valid = [i for i, b in enumerate(ph_bits) if b is not None]
    packed = {i: np.packbits(ph_bits[i]) for i in valid}
    thr = CONFIG["near_dup_hamming_max"]

    # Band the 64 bits into 4 blocks of 16. Two hashes within 5 bits of each other must agree
    # exactly on at least one block when thr < n_bands, so this candidate generation is lossless
    # for thr <= 3 and near-lossless above; we additionally sweep all pairs when n is small.
    POPCNT = np.array([bin(x).count("1") for x in range(256)], dtype=np.uint8)

    def hamming(i, j):
        return int(POPCNT[np.bitwise_xor(packed[i], packed[j])].sum())

    checked = set()
    if len(valid) <= 3000:
        for a_pos in range(len(valid)):
            i = valid[a_pos]
            for j in valid[a_pos + 1:]:
                if hamming(i, j) <= thr:
                    near_pairs.append((i, j))
    else:
        buckets = defaultdict(list)
        for i in valid:
            bits = ph_bits[i]
            for b in range(4):
                key = (b, bits_to_hex(np.concatenate([bits[b*16:(b+1)*16],
                                                      np.zeros(48, dtype=bool)])))
                buckets[key].append(i)
        for key, members in buckets.items():
            if len(members) < 2 or len(members) > 400:
                continue
            for a_pos in range(len(members)):
                for j in members[a_pos + 1:]:
                    i = members[a_pos]
                    pair = (min(i, j), max(i, j))
                    if pair in checked:
                        continue
                    checked.add(pair)
                    if hamming(*pair) <= thr:
                        near_pairs.append(pair)

    near_pairs = [(i, j) for i, j in near_pairs if CLEAN.loc[i, "sha256"] != CLEAN.loc[j, "sha256"]]
    for i, j in near_pairs:
        union(i, j)

print(f"Near-duplicate pairs (Hamming <= {CONFIG['near_dup_hamming_max']} of 64): {len(near_pairs)}")
cross_label_near = [(i, j) for i, j in near_pairs if CLEAN.loc[i, "class"] != CLEAN.loc[j, "class"]]
print(f"  of which span two different classes: {len(cross_label_near)}")
for i, j in cross_label_near[:15]:
    print(f"   {CLEAN.loc[i,'class']}/{CLEAN.loc[i,'filename']}  ~  "
          f"{CLEAN.loc[j,'class']}/{CLEAN.loc[j,'filename']}")

CLEAN["dup_group"] = [find(i) for i in range(len(CLEAN))]
group_sizes = CLEAN["dup_group"].value_counts()
multi = group_sizes[group_sizes > 1]
print(f"\nConnected duplicate components : {len(multi)}")
print(f"Images inside such a component : {int(multi.sum())}")
print(f"Distinct groups to split on    : {CLEAN['dup_group'].nunique()} "
      f"(vs {len(CLEAN)} images)")
print("\nLeakage assessment: the split in Section 07 assigns whole dup_group components, so no "
      "image and no near-identical variant can appear in two different splits.")

# ---- Guard against an over-merged duplicate graph --------------------------------------------
# Near-duplicate edges are transitive through the union-find: A~B and B~C merges A, B and C even
# when A and C are not similar. With a threshold that is too loose this snowballs into one giant
# component, which would then have to land entirely in one split and would wreck stratification.
largest = int(group_sizes.iloc[0]) if len(group_sizes) else 0
largest_share = largest / max(len(CLEAN), 1)
print(f"\nLargest component              : {largest} images ({largest_share*100:.1f}% of corpus)")
print("Top component sizes            :", group_sizes.head(8).tolist())
if largest_share > 0.02:
    print("\n!! WARNING: one component holds "
          f"{largest_share*100:.1f}% of the images. That is far more than genuine re-posts usually "
          "produce and points at a perceptual threshold too loose for this corpus; flat, "
          "low-texture or heavily letterboxed images give degenerate perceptual hashes. Inspect "
          "the pairs plotted below. If they are not actually the same meme, lower "
          "CONFIG['near_dup_hamming_max'] (try 3, then 2) and re-run Section 05 before splitting.")
if largest_share > 0.20:
    raise RuntimeError(
        f"Refusing to continue: the largest duplicate component covers {largest_share*100:.1f}% of "
        "the corpus, so a grouped split cannot produce usable validation and test sets. Lower "
        "CONFIG['near_dup_hamming_max'], or set CONFIG['run_near_dup'] = False to group on exact "
        "duplicates only, then re-run Section 05.")

dup_report = {
    "exact_duplicate_groups": len(exact_groups),
    "redundant_exact_copies": int(n_exact_extra),
    "exact_dups_spanning_classes": len(cross_label_exact),
    "near_duplicate_pairs": len(near_pairs),
    "near_dups_spanning_classes": len(cross_label_near),
    "connected_components_gt1": int(len(multi)),
    "largest_component_images": largest,
    "largest_component_share": round(float(largest_share), 4),
    "images_in_components_gt1": int(multi.sum()),
    "distinct_groups": int(CLEAN["dup_group"].nunique()),
    "hamming_threshold": CONFIG["near_dup_hamming_max"],
}

### 05b. Visual check of detected duplicate pairs

A handful of detected pairs are displayed so you can confirm the perceptual threshold is doing what
you expect. If these look like genuinely different memes, lower `near_dup_hamming_max` and re-run
Section 05 before splitting.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
matplotlib.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
    "font.size": 10, "axes.titlesize": 11, "axes.labelsize": 10,
    "axes.grid": True, "grid.alpha": 0.25, "axes.spines.top": False, "axes.spines.right": False,
})
sns.set_palette([CLASS_COLORS[c] for c in CLASSES])
PLOTS = OUT / "plots"

def save_fig(fig, name):
    path = PLOTS / f"{name}.png"
    fig.savefig(path)
    print("saved", path)

show_pairs = (near_pairs[:4] if near_pairs
              else [(g[0], g[1]) for g in exact_groups[:4]] if exact_groups else [])
if show_pairs:
    fig, axes = plt.subplots(len(show_pairs), 2, figsize=(6, 3 * len(show_pairs)))
    axes = np.atleast_2d(axes)
    for row, (i, j) in enumerate(show_pairs):
        for col, idx in enumerate([i, j]):
            ax = axes[row, col]
            try:
                ax.imshow(Image.open(CLEAN.loc[idx, "filepath"]).convert("RGB"))
            except Exception:
                ax.text(0.5, 0.5, "unreadable", ha="center")
            ax.set_title(f"{CLEAN.loc[idx,'class']}\n{CLEAN.loc[idx,'filename'][:28]}", fontsize=8)
            ax.axis("off")
    fig.suptitle("Detected duplicate / near-duplicate pairs", y=1.0)
    fig.tight_layout(); save_fig(fig, "05_duplicate_pairs"); plt.show()
else:
    print("No duplicate or near-duplicate pairs were detected, so there is nothing to display.")

---
## 06. Exploratory Data Analysis

Six figures, all written to `results/plots/`. The sample grid uses the **same number of examples per
class** so it does not visually over-represent the majority class.

In [ ]:
# A. Class distribution -----------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(7, 4))
counts = dist["n_images"].values
bars = ax.bar(CLASSES, counts, color=[CLASS_COLORS[c] for c in CLASSES], edgecolor="black", lw=0.6)
for b, v, pct in zip(bars, counts, dist["percentage"].values):
    ax.text(b.get_x() + b.get_width() / 2, v, f"{v}\n({pct:.1f}%)",
            ha="center", va="bottom", fontsize=9)
ax.set_ylabel("Number of images"); ax.set_xlabel("Class")
ax.set_title(f"Class distribution (n = {int(counts.sum())} decodable images)")
ax.set_ylim(0, counts.max() * 1.18)
fig.tight_layout(); save_fig(fig, "06a_class_distribution"); plt.show()

In [ ]:
# B. Class percentage (donut) -----------------------------------------------------------------
fig, ax = plt.subplots(figsize=(5.5, 5.5))
w, t, a = ax.pie(counts, labels=CLASSES, autopct="%1.1f%%", startangle=90,
                 colors=[CLASS_COLORS[c] for c in CLASSES],
                 wedgeprops=dict(width=0.42, edgecolor="white"))
ax.set_title("Class share of the corpus")
ax.text(0, 0, f"{int(counts.sum())}\nimages", ha="center", va="center", fontsize=12)
fig.tight_layout(); save_fig(fig, "06b_class_share"); plt.show()

In [ ]:
# C. Image dimensions -------------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(CLEAN["width"], bins=40, color="#4C72B0", edgecolor="white")
axes[0].set_title("Width (px)"); axes[0].set_xlabel("width"); axes[0].set_ylabel("count")
axes[1].hist(CLEAN["height"], bins=40, color="#DD8452", edgecolor="white")
axes[1].set_title("Height (px)"); axes[1].set_xlabel("height")
sc = axes[2].scatter(CLEAN["width"], CLEAN["height"], s=8, alpha=0.45,
                     c=[CLASS_COLORS[c] for c in CLEAN["class"]])
axes[2].set_title("Width vs height"); axes[2].set_xlabel("width"); axes[2].set_ylabel("height")
handles = [plt.Line2D([], [], marker="o", ls="", color=CLASS_COLORS[c], label=c) for c in CLASSES]
axes[2].legend(handles=handles, fontsize=7, loc="upper left")
fig.suptitle("Image dimension distribution")
fig.tight_layout(); save_fig(fig, "06c_dimensions"); plt.show()

print("Min dimensions  : {}x{}".format(int(CLEAN['width'].min()), int(CLEAN['height'].min())))
print("Max dimensions  : {}x{}".format(int(CLEAN['width'].max()), int(CLEAN['height'].max())))
print("Mean dimensions : {:.1f}x{:.1f}".format(CLEAN['width'].mean(), CLEAN['height'].mean()))
print("Median          : {}x{}".format(int(CLEAN['width'].median()), int(CLEAN['height'].median())))

In [ ]:
# D. Aspect ratio -----------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(CLEAN["aspect_ratio"], bins=50, color="#55A868", edgecolor="white")
axes[0].axvline(1.0, ls="--", c="k", lw=1, label="square")
axes[0].set_xlabel("aspect ratio (w/h)"); axes[0].set_ylabel("count")
axes[0].set_title("Aspect-ratio distribution"); axes[0].legend()
sns.boxplot(data=CLEAN, x="class", y="aspect_ratio", order=CLASSES,
            palette=[CLASS_COLORS[c] for c in CLASSES], ax=axes[1])
axes[1].axhline(1.0, ls="--", c="k", lw=1)
axes[1].set_title("Aspect ratio per class"); axes[1].set_xlabel("")
fig.tight_layout(); save_fig(fig, "06d_aspect_ratio"); plt.show()

portrait = int((CLEAN["aspect_ratio"] < 0.95).sum())
square   = int(((CLEAN["aspect_ratio"] >= 0.95) & (CLEAN["aspect_ratio"] <= 1.05)).sum())
landscape= int((CLEAN["aspect_ratio"] > 1.05).sum())
print(f"portrait: {portrait} | near-square: {square} | landscape: {landscape}")

In [ ]:
# E. File size ---------------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(CLEAN["file_size_kb"], bins=50, color="#C44E52", edgecolor="white")
axes[0].set_xlabel("file size (KB)"); axes[0].set_ylabel("count")
axes[0].set_title("File-size distribution")
sns.violinplot(data=CLEAN, x="class", y="file_size_kb", order=CLASSES, cut=0,
               palette=[CLASS_COLORS[c] for c in CLASSES], ax=axes[1])
axes[1].set_title("File size per class"); axes[1].set_xlabel(""); axes[1].set_ylabel("KB")
fig.tight_layout(); save_fig(fig, "06e_file_size"); plt.show()
print(CLEAN.groupby("class")["file_size_kb"].describe()[["count","mean","50%","min","max"]]
      .reindex(CLASSES).round(1).to_string())

In [ ]:
# F. Balanced sample grid ----------------------------------------------------------------------
N_PER_CLASS = 5
rng = np.random.default_rng(SEED)
fig, axes = plt.subplots(NUM_CLASSES, N_PER_CLASS, figsize=(2.3 * N_PER_CLASS, 2.5 * NUM_CLASSES))
for r, cls in enumerate(CLASSES):
    pool = CLEAN.index[CLEAN["class"] == cls].to_numpy()
    pick = rng.choice(pool, size=min(N_PER_CLASS, len(pool)), replace=False)
    for c in range(N_PER_CLASS):
        ax = axes[r, c]; ax.axis("off")
        if c < len(pick):
            try:
                ax.imshow(Image.open(CLEAN.loc[pick[c], "filepath"]).convert("RGB"))
            except Exception:
                ax.text(0.5, 0.5, "unreadable", ha="center", va="center")
        if c == 0:
            ax.set_title(cls, loc="left", color=CLASS_COLORS[cls], fontweight="bold", fontsize=12)
fig.suptitle(f"Representative samples ({N_PER_CLASS} per class, equal count per row)", y=1.002)
fig.tight_layout(); save_fig(fig, "06f_sample_grid"); plt.show()

---
## 07. Train / Validation / Test Split

**Stratified by class, grouped by duplicate component, seeded.**

Standard `train_test_split(stratify=y)` is unsafe here because it would break duplicate components
apart. Instead we sort the duplicate components of each class, shuffle them with the fixed seed, and
greedily fill train, then validation, then test to their target image counts. Since the vast
majority of components hold exactly one image, stratification stays tight; the exact realised
percentages are printed rather than assumed.

A component containing more than one class (a cross-label duplicate found in Section 05) is
assigned by its majority class and counted once, which is the conservative choice: it removes the
leakage without discarding labelled data.

The default 70/15/15 is kept when every class can still put at least 10 images in validation and in
test. When a class is too small for that, the cell says so and falls back to 80/10/10 with an
explicit warning, because a 5-image validation set cannot support model selection.

In [ ]:
# Component -> (majority class, size)
comp_rows = []
for gid, sub in CLEAN.groupby("dup_group"):
    maj = sub["class"].value_counts().idxmax()
    comp_rows.append({"dup_group": gid, "class": maj, "size": len(sub),
                      "mixed_label": sub["class"].nunique() > 1})
comp_df = pd.DataFrame(comp_rows)
print(f"Components: {len(comp_df)}  (mixed-label: {int(comp_df['mixed_label'].sum())})")

ratios = CONFIG["split"]
min_per_class = dist["n_images"].min()
if min_per_class * ratios["val"] < 10 or min_per_class * ratios["test"] < 10:
    print(f"!! Smallest class has {min_per_class} images; {ratios['val']:.0%} of it is "
          f"{min_per_class*ratios['val']:.1f} validation images, too few for stable model "
          f"selection. Falling back to 80/10/10 and reporting test metrics with the caveat that "
          f"the per-class test support is small.")
    ratios = {"train": 0.80, "val": 0.10, "test": 0.10}
print("Split ratios in use:", ratios)

rng = np.random.default_rng(SEED)
assign = {}
for cls in CLASSES:
    sub = comp_df[comp_df["class"] == cls].sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    n_img = int(sub["size"].sum())
    target = {"train": ratios["train"] * n_img, "val": ratios["val"] * n_img,
              "test": ratios["test"] * n_img}
    filled = {"train": 0, "val": 0, "test": 0}
    # Largest components first so one big blob cannot overshoot a small split at the end.
    for _, row in sub.sort_values("size", ascending=False).iterrows():
        deficit = {s: (target[s] - filled[s]) / max(target[s], 1e-9) for s in target}
        chosen = max(deficit, key=deficit.get)
        assign[row["dup_group"]] = chosen
        filled[chosen] += row["size"]
    print(f"  {cls:<12} components={len(sub):<5} images={n_img:<5} -> "
          f"train {filled['train']}, val {filled['val']}, test {filled['test']}")
    if n_img == 0:
        print(f"      note: every {cls} image sits in a component whose majority class is a "
              "different class, so they are counted under that class on the line above. The "
              "authoritative per-split class counts are in the table printed by the next cell.")

CLEAN["split"] = CLEAN["dup_group"].map(assign)
assert CLEAN["split"].notna().all(), "some image was never assigned a split"

In [ ]:
# ---- Leakage assertions (these must pass before any training happens) ------------------------
g2s = CLEAN.groupby("dup_group")["split"].nunique()
assert (g2s == 1).all(), "a duplicate component was split across two splits"

sha2s = CLEAN.groupby("sha256")["split"].nunique()
assert (sha2s == 1).all(), "an exact duplicate appears in two splits"

paths_by_split = {s: set(g["filepath"]) for s, g in CLEAN.groupby("split")}
for a in paths_by_split:
    for b in paths_by_split:
        if a < b:
            assert not (paths_by_split[a] & paths_by_split[b]), f"path overlap {a}/{b}"
print("Leakage checks passed: no component, no byte-identical file and no path is shared "
      "between splits.")

split_tab = (CLEAN.groupby(["split", "class"]).size().unstack(fill_value=0)
             .reindex(index=["train", "val", "test"], columns=CLASSES).fillna(0).astype(int))
split_tab["TOTAL"] = split_tab.sum(axis=1)
print("\nAbsolute counts:")
print(split_tab.to_string())
print("\nWithin-split class percentages:")
pct_tab = (split_tab[CLASSES].div(split_tab["TOTAL"], axis=0) * 100).round(2)
print(pct_tab.to_string())
print("\nShare of each class landing in each split (%):")
class_share = (split_tab[CLASSES] / split_tab[CLASSES].sum(axis=0) * 100).round(2)
print(class_share.to_string())

# Every class must appear in every split; otherwise macro-F1 and the confusion matrix are
# undefined for that class and model selection cannot see it.
empty = [(sp, c) for sp in ["train", "val", "test"] for c in CLASSES if split_tab.loc[sp, c] == 0]
if empty:
    raise RuntimeError(
        "These (split, class) cells are empty: " + ", ".join(f"{sp}/{c}" for sp, c in empty) +
        ". The usual cause is a duplicate component that absorbed most of a class (see the "
        "component sizes printed in Section 05); the other is a class with too few images for the "
        "chosen ratios. Fix by lowering CONFIG['near_dup_hamming_max'], or by adjusting "
        "CONFIG['split'], then re-run Sections 05 to 07.")

train_df = CLEAN[CLEAN["split"] == "train"].reset_index(drop=True)
val_df   = CLEAN[CLEAN["split"] == "val"].reset_index(drop=True)
test_df  = CLEAN[CLEAN["split"] == "test"].reset_index(drop=True)
print(f"\ntrain={len(train_df)}  val={len(val_df)}  test={len(test_df)}")

CLEAN.to_csv(OUT / "splits" / "dataset_index.csv", index=False)
for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    d.to_csv(OUT / "splits" / f"{name}.csv", index=False)
print("Split index written to", OUT / "splits")

In [ ]:
# ---- Split visualisation ---------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
x = np.arange(NUM_CLASSES); w = 0.26
for k, s in enumerate(["train", "val", "test"]):
    axes[0].bar(x + (k - 1) * w, split_tab.loc[s, CLASSES].values, w, label=s)
axes[0].set_xticks(x); axes[0].set_xticklabels(CLASSES, rotation=20)
axes[0].set_ylabel("images"); axes[0].set_title("Counts per class per split"); axes[0].legend()

bottom = np.zeros(3)
for cls in CLASSES:
    vals = pct_tab[cls].values
    axes[1].bar(["train", "val", "test"], vals, bottom=bottom, color=CLASS_COLORS[cls], label=cls)
    bottom += vals
axes[1].set_ylabel("% of split"); axes[1].set_title("Class composition within each split")
axes[1].legend(fontsize=7, ncol=2)

axes[2].bar(["train", "val", "test"], split_tab["TOTAL"].values,
            color=["#4C72B0", "#DD8452", "#55A868"], edgecolor="black", lw=0.6)
for i, v in enumerate(split_tab["TOTAL"].values):
    axes[2].text(i, v, f"{v}\n({v/len(CLEAN)*100:.1f}%)", ha="center", va="bottom")
axes[2].set_title("Split sizes"); axes[2].set_ylabel("images")
axes[2].set_ylim(0, split_tab["TOTAL"].max() * 1.2)
fig.tight_layout(); save_fig(fig, "07_split_distribution"); plt.show()

---
## 08. Preprocessing & Augmentation

Memes are not natural photographs. Their meaning sits in overlaid Bangla/English text, in
recognisable faces of public figures, in party symbols and in team logos. Every augmentation below
is chosen so that none of those survive-or-die cues get destroyed.

| Transform | Setting | Why |
|---|---|---|
| Aspect-preserving resize into the pixel budget | always | Qwen-VL handles native aspect ratios; we only cap the pixel count so the visual-token count stays inside the VRAM budget. |
| Horizontal flip | **off** (`hflip_p = 0.0`) | A mirrored meme has mirrored text. Bangla and English script become unreadable, which removes exactly the signal the model needs. Turn it on only if your corpus is dominated by text-free image macros. |
| Rotation +/- 5 deg, p = 0.3 | mild | Simulates screenshot and re-photographing artefacts common in shared memes, while leaving text legible. Larger angles start to clip the caption bars. |
| Colour jitter (brightness/contrast/saturation 0.12/0.12/0.08), p = 0.3 | mild | Covers the compression and screenshot-brightness variance of re-shared memes. Saturation is kept lowest because party colours and team jerseys are class evidence. |
| Aspect-preserving random crop, 88-100% of the area, p = 0.25 | gentle | Simulates cropped re-posts. The 88% floor keeps whole caption bars in frame; aggressive cropping would silently delete the caption and leave the sample mislabelled. The aspect ratio is held fixed so training images stay geometrically like evaluation images. |
| Vertical flip, heavy rotation, grayscale, cutout, mixup | **not used** | Each destroys text, faces, symbols or logos, i.e. the class evidence itself. |

Validation and test go through the deterministic path only: convert to RGB, cap pixels, no
randomness. This is asserted in code below.

In [ ]:
from torchvision import transforms as T

AUG = CONFIG["aug"]

def to_rgb(img: Image.Image) -> Image.Image:
    """Flatten alpha onto white and force 3 channels. Animated files use their first frame."""
    if getattr(img, "is_animated", False):
        img.seek(0)
    if img.mode in ("RGBA", "LA", "PA"):
        bg = Image.new("RGB", img.size, (255, 255, 255))
        bg.paste(img.convert("RGBA"), mask=img.convert("RGBA").split()[-1])
        return bg
    return img.convert("RGB")

def fit_pixel_budget(img: Image.Image, max_pixels: int, min_pixels: int) -> Image.Image:
    """Scale so total pixels fall inside [min_pixels, max_pixels], preserving aspect ratio."""
    w, h = img.size
    n = w * h
    if n > max_pixels:
        s = math.sqrt(max_pixels / n)
    elif n < min_pixels:
        s = math.sqrt(min_pixels / n)
    else:
        return img
    nw, nh = max(28, int(w * s)), max(28, int(h * s))
    return img.resize((nw, nh), Image.Resampling.LANCZOS)

def random_area_crop(img: Image.Image) -> Image.Image:
    """Crop a random sub-rectangle covering `crop_scale` of the area, keeping the aspect ratio,
    then resize back to the original size. Aspect ratio is preserved deliberately: Qwen-VL encodes
    the native ratio, and squashing a wide caption bar into a square is a distortion the evaluation
    path would never produce."""
    lo, hi = AUG["crop_scale"]
    area_frac = random.uniform(lo, hi)
    s = math.sqrt(area_frac)
    w, h = img.size
    nw, nh = max(16, int(w * s)), max(16, int(h * s))
    x0 = random.randint(0, w - nw)
    y0 = random.randint(0, h - nh)
    return img.crop((x0, y0, x0 + nw, y0 + nh)).resize((w, h), Image.Resampling.LANCZOS)

train_aug = T.Compose([
    T.RandomApply([T.Lambda(random_area_crop)], p=AUG["random_resized_crop_p"]),
    T.RandomApply([T.RandomRotation(AUG["rotation_deg"], expand=False, fill=(255, 255, 255))],
                  p=AUG["rotation_p"]),
    T.RandomApply([T.ColorJitter(brightness=AUG["brightness"], contrast=AUG["contrast"],
                                 saturation=AUG["saturation"])], p=AUG["jitter_p"]),
    T.RandomHorizontalFlip(p=AUG["hflip_p"]),
])

def load_image(path: str, train: bool) -> Image.Image:
    with Image.open(path) as im:
        img = to_rgb(im)
    if train:
        img = train_aug(img)
    return fit_pixel_budget(img, CONFIG["max_pixels"], CONFIG["min_pixels"])

# Determinism check on the evaluation path: same file twice must give identical pixels.
_p = val_df["filepath"].iloc[0] if len(val_df) else CLEAN["filepath"].iloc[0]
_a, _b = load_image(_p, train=False), load_image(_p, train=False)
assert np.array_equal(np.asarray(_a), np.asarray(_b)), "eval path is not deterministic"
print("Eval preprocessing is deterministic (no augmentation on val/test). OK")
print("Eval image size for", Path(_p).name, "->", _a.size)

In [ ]:
# ---- Visual sanity check of the augmentation pipeline ----------------------------------------
demo_path = train_df["filepath"].iloc[0]
fig, axes = plt.subplots(1, 6, figsize=(16, 3))
axes[0].imshow(load_image(demo_path, train=False)); axes[0].set_title("eval (deterministic)")
axes[0].axis("off")
for k in range(1, 6):
    set_seed(SEED + k)
    axes[k].imshow(load_image(demo_path, train=True)); axes[k].set_title(f"train draw {k}")
    axes[k].axis("off")
set_seed(SEED)
fig.suptitle("Augmentation preview: text, faces and symbols must stay readable")
fig.tight_layout(); save_fig(fig, "08_augmentation_preview"); plt.show()

---
## 09. Qwen-VL Setup

### The classification formulation

Qwen-VL is a generative vision-language model. We keep it generative and add no new head:

* **Input**: the meme image plus one fixed instruction.
* **Target**: the class name as a single short string, e.g. `Political` or `Neutral`.
* **Training loss**: next-token cross-entropy computed **only on the answer tokens**.
* **Inference**: exact restricted scoring over the candidate class tokens.

In [ ]:
import transformers
from transformers import AutoProcessor, AutoConfig

print("transformers", transformers.__version__)

def find_local_model_dir():
    """Look for an offline Qwen-VL checkpoint attached as a Kaggle dataset."""
    base = Path("/kaggle/input")
    if not base.exists():
        return None
    for cfg in base.glob("*/**/config.json"):
        try:
            with open(cfg) as f:
                j = json.load(f)
        except Exception:
            continue
        arch = " ".join(j.get("architectures", []) or []) + " " + str(j.get("model_type", ""))
        if "qwen" in arch.lower() and "vl" in arch.lower():
            print("Found local Qwen-VL checkpoint:", cfg.parent)
            return str(cfg.parent)
    return None

LOCAL_DIR = find_local_model_dir()
MODEL_SOURCE = LOCAL_DIR or CONFIG["model_id"]
if LOCAL_DIR is None and not INTERNET:
    print("Warning: No local Qwen-VL checkpoint and no Internet. Model download will require Internet enabled.")
print("Model source:", MODEL_SOURCE)

try:
    cfg = AutoConfig.from_pretrained(MODEL_SOURCE, trust_remote_code=True)
    print("architectures:", getattr(cfg, "architectures", None))
    print("model_type   :", getattr(cfg, "model_type", None))
except Exception as e:
    print("Note: AutoConfig failed without internet / local weights:", e)

In [ ]:
# ---- Precision and quantisation decided from the actual GPU ---------------------------------
BF16_OK = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
COMPUTE_DTYPE = torch.bfloat16 if BF16_OK else torch.float16
TOTAL_VRAM = (torch.cuda.get_device_properties(0).total_memory / 1024**3
              if torch.cuda.is_available() else 0.0)

HAS_BNB = False
try:
    import bitsandbytes
    HAS_BNB = True
except Exception:
    pass

USE_4BIT = bool(HAS_BNB and TOTAL_VRAM and TOTAL_VRAM < 15.5)

print(f"VRAM {TOTAL_VRAM:.1f} GiB | bf16 supported: {BF16_OK} | compute dtype: {COMPUTE_DTYPE}")
print(f"bitsandbytes available: {HAS_BNB} | loading in 4-bit (QLoRA): {USE_4BIT}")
if not BF16_OK:
    print("  -> fp16 path: a GradScaler is used and the LoRA parameters are kept in fp32.")


In [ ]:
# ---- Processor --------------------------------------------------------------------------------
try:
    processor = AutoProcessor.from_pretrained(
        MODEL_SOURCE, trust_remote_code=True,
        min_pixels=CONFIG["min_pixels"], max_pixels=CONFIG["max_pixels"])
    print("Processor loaded with an explicit min_pixels / max_pixels budget.")
except (TypeError, ValueError) as e:
    print("This image processor rejected min_pixels/max_pixels "
          f"({type(e).__name__}); loading with its defaults instead.")
    processor = AutoProcessor.from_pretrained(MODEL_SOURCE, trust_remote_code=True)
# Either way the budget is also enforced on our side by fit_pixel_budget() in Section 08, which
# resizes every image before it reaches the processor. The kwargs above are belt and braces.
tokenizer = processor.tokenizer
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
PAD_ID = tokenizer.pad_token_id
print("pad_token_id:", PAD_ID, "| eos_token_id:", tokenizer.eos_token_id)
print("image processor:", type(processor.image_processor).__name__)

### 09b. The prompt

One prompt string, defined once, used everywhere. It lists the five classes in the canonical order
and asks for exactly one of them. It contains no example answers and no hint about the true label.

In [ ]:
class_list_str = ", ".join(CLASSES)
CLASSIFICATION_PROMPT = (
    f"You are analysing a meme circulated on Bangladeshi social media.\n"
    f"Classify this meme into exactly one of the following categories:\n"
    f"{class_list_str}.\n"
    f"Answer with the category name only."
)

def build_messages(image):
    return [{"role": "user", "content": [{"type": "image", "image": image},
                                         {"type": "text", "text": CLASSIFICATION_PROMPT}]}]

def prompt_text_for(image):
    """Chat-template string ending at the point where the assistant answer begins."""
    return processor.apply_chat_template(build_messages(image), tokenize=False,
                                         add_generation_prompt=True)

_demo_img = load_image(train_df["filepath"].iloc[0], train=False)
_pt = prompt_text_for(_demo_img)
print(_pt)
print("-" * 70)
print("Full training sequence ends with the label plus EOS, e.g.:")
print(repr(_pt[-80:] + CLASSES[0] + tokenizer.eos_token))

In [ ]:
# ---- Label token ids and the scoring strategy -------------------------------------------------
LABEL_TOKEN_IDS = {c: tokenizer(c, add_special_tokens=False)["input_ids"] for c in CLASSES}
for c, ids in LABEL_TOKEN_IDS.items():
    print(f"{c:<12} -> {ids}  {tokenizer.convert_ids_to_tokens(ids)}")

FIRST_TOKENS = [ids[0] for ids in LABEL_TOKEN_IDS.values()]
FAST_SCORING = len(set(FIRST_TOKENS)) == NUM_CLASSES
print("\nFirst tokens distinct across the five labels:", FAST_SCORING)
print("Scoring strategy:",
      "single forward pass, distribution over the five first tokens (exact, since the labels are "
      "already separated at position 1)" if FAST_SCORING else
      "five scored continuations per image, summed token log-probabilities")
FIRST_TOKEN_TENSOR = torch.tensor(FIRST_TOKENS, dtype=torch.long)

### 09c. Dataset and collator

`MemeDataset` returns a processed sample. The collator pads to the batch maximum, concatenates the
visual tensors along the patch axis the way Qwen-VL expects, and builds the `-100` label mask.
`prompt_len` is measured by running the processor on the prompt alone **with the same image**, so
the visual-token expansion is identical and the prefix boundary is exact.

In [ ]:
from torch.utils.data import Dataset, DataLoader

class MemeDataset(Dataset):
    def __init__(self, df, train=False, with_answer=True):
        self.df = df.reset_index(drop=True)
        self.train = train
        self.with_answer = with_answer

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        img = load_image(r["filepath"], train=self.train)
        msgs = build_messages(img)
        prefix = processor.apply_chat_template(msgs, tokenize=False,
                                             add_generation_prompt=True)
        if self.with_answer:
            full_text = prefix + r["class"] + tokenizer.eos_token
            proc_p = processor(text=[prefix], images=[img], return_tensors="pt")
            prompt_len = proc_p["input_ids"].shape[1]
            proc_full = processor(text=[full_text], images=[img], return_tensors="pt")
            item = {}
            for k, v in proc_full.items():
                if k in ("input_ids", "attention_mask"):
                    item[k] = v.squeeze(0)
                elif k == "image_grid_thw":
                    item[k] = v if v.dim() == 2 else v.view(-1, 3)
                else:
                    item[k] = v.squeeze(0) if v.dim() > 2 else v
            item["prompt_len"] = prompt_len
        else:
            proc = processor(text=[prefix], images=[img], return_tensors="pt")
            item = {}
            for k, v in proc.items():
                if k in ("input_ids", "attention_mask"):
                    item[k] = v.squeeze(0)
                elif k == "image_grid_thw":
                    item[k] = v if v.dim() == 2 else v.view(-1, 3)
                else:
                    item[k] = v.squeeze(0) if v.dim() > 2 else v
        item["row_index"] = idx
        item["label_id"] = int(r["label_id"])
        return item

# Visual fields produced by Qwen-VL processors. They must be concatenated across images
# along their leading axis.
VISUAL_KEYS = ("pixel_values", "pixel_values_videos",
               "image_grid_thw", "video_grid_thw")

def collate(batch):
    pad_side = getattr(tokenizer, "padding_side", "right")
    L = max(b["input_ids"].shape[0] for b in batch)
    input_ids = torch.full((len(batch), L), PAD_ID, dtype=torch.long)
    attn = torch.zeros((len(batch), L), dtype=torch.long)
    labels = torch.full((len(batch), L), -100, dtype=torch.long)
    for i, b in enumerate(batch):
        n = b["input_ids"].shape[0]
        sl = slice(0, n) if pad_side == "right" else slice(L - n, L)
        input_ids[i, sl] = b["input_ids"]
        attn[i, sl] = b["attention_mask"]
        if "prompt_len" in b:
            lab = b["input_ids"].clone()
            lab[: b["prompt_len"]] = -100        # mask instruction + image tokens
            labels[i, sl] = lab
    out = {"input_ids": input_ids, "attention_mask": attn}
    if any("prompt_len" in b for b in batch):
        out["labels"] = labels
    for k in VISUAL_KEYS:
        if k in batch[0]:
            tensors = [b[k] for b in batch]
            if k == "image_grid_thw":
                tensors = [t.view(-1, 3) if t.dim() < 2 else t for t in tensors]
                out[k] = torch.cat(tensors, dim=0)
            elif k == "pixel_values":
                out[k] = torch.cat(tensors, dim=0)
            else:
                out[k] = torch.cat(tensors, dim=0)
    out["row_index"] = torch.tensor([b["row_index"] for b in batch])
    out["label_id"] = torch.tensor([b["label_id"] for b in batch])
    return out

_probe = MemeDataset(train_df.head(2), train=False, with_answer=True)
_b = collate([_probe[0], _probe[1]])
print({k: (tuple(v.shape) if torch.is_tensor(v) else v) for k, v in _b.items()})
n_sup = int((_b["labels"] != -100).sum())
print(f"Supervised target tokens in this 2-sample batch: {n_sup} "
      f"(answer tokens + EOS only; everything else is -100)")
print("Decoded targets:",
      [tokenizer.decode(_b['labels'][i][_b['labels'][i] != -100]) for i in range(2)])


---
## 10. LoRA Configuration

LoRA adapters are attached to the attention projections and the MLP projections of the **language**
decoder. The vision tower and the vision-to-language merger stay frozen by default: the visual
encoder already produces good general features, the corpus here is small, and freezing it keeps both
VRAM and the risk of catastrophic forgetting down. Set `adapt_vision_tower = True` in the config to
include the visual blocks as well.

Module names are not assumed. We enumerate the actual `nn.Linear` modules in the loaded model,
print the naming scheme, and intersect it with the configured target list, so the cell fails loudly
rather than silently training nothing if the checkpoint uses different names.

In [ ]:
from transformers import BitsAndBytesConfig

def load_base_model():
    kwargs = dict(dtype=COMPUTE_DTYPE, device_map="auto" if torch.cuda.is_available() else None,
                  trust_remote_code=True, attn_implementation=CONFIG["attn_implementation"])
    if USE_4BIT and HAS_BNB and torch.cuda.is_available():
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=COMPUTE_DTYPE, bnb_4bit_use_double_quant=True)
    
    loaders = [
        ("Qwen2_5_VLForConditionalGeneration", lambda: getattr(transformers, "Qwen2_5_VLForConditionalGeneration", None)),
        ("Qwen2VLForConditionalGeneration", lambda: getattr(transformers, "Qwen2VLForConditionalGeneration", None)),
        ("Qwen3VLForConditionalGeneration", lambda: getattr(transformers, "Qwen3VLForConditionalGeneration", None)),
        ("AutoModelForImageTextToText", lambda: getattr(transformers, "AutoModelForImageTextToText", None)),
        ("AutoModelForVision2Seq", lambda: getattr(transformers, "AutoModelForVision2Seq", None)),
        ("AutoModelForCausalLM", lambda: getattr(transformers, "AutoModelForCausalLM", None)),
    ]
    
    last_err = None
    for name, get_cls in loaders:
        cls = get_cls()
        if cls is not None:
            try:
                print(f"Attempting to load model using {name}...")
                return cls.from_pretrained(MODEL_SOURCE, **kwargs)
            except Exception as e:
                last_err = e
                print(f"{name} failed ({type(e).__name__}): {e}")
    raise RuntimeError(f"Could not load {MODEL_SOURCE}. Last error: {last_err}")

t0 = time.time()
base_model = load_base_model()
print(f"Loaded {type(base_model).__name__} in {time.time()-t0:.0f}s")
print("dtype:", next(base_model.parameters()).dtype, "| device map:",
      getattr(base_model, "hf_device_map", "single device"))
DEVICE = next(base_model.parameters()).device


In [ ]:
# ---- Inspect the module naming before choosing LoRA targets ----------------------------------
linear_names = defaultdict(int)
vision_markers = ("visual", "vision_tower", "vision_model")
for name, mod in base_model.named_modules():
    if isinstance(mod, torch.nn.Linear) or mod.__class__.__name__ in ("Linear4bit", "Linear8bitLt"):
        leaf = name.split(".")[-1]
        side = "vision" if any(m in name for m in vision_markers) else "language"
        linear_names[(side, leaf)] += 1

print("Linear module leaf names (side, name -> count):")
for (side, leaf), n in sorted(linear_names.items(), key=lambda x: (-x[1], x[0])):
    print(f"  {side:<8} {leaf:<16} {n}")

wanted = set(CONFIG["lora"]["target_modules"])
present_lang = {leaf for (side, leaf) in linear_names if side == "language"}
present_vis = {leaf for (side, leaf) in linear_names if side == "vision"}
targets = sorted(wanted & present_lang)
missing_targets = sorted(wanted - present_lang)
print("\nLoRA targets resolved in the language decoder:", targets)
if missing_targets:
    print("Configured but absent (ignored):", missing_targets)
if not targets:
    raise RuntimeError("None of the configured LoRA target names exist in this checkpoint. "
                       "Pick names from the list printed above.")
# Build the exclusion regex from the module names this checkpoint actually uses, rather than
# assuming the vision stack is called "visual". PEFT matches a string exclude_modules with
# re.fullmatch against the full module key.
vision_prefixes = sorted({m for m in vision_markers
                          if any(m in name for name, _ in base_model.named_modules())})
EXCLUDE_REGEX = ".*(" + "|".join(vision_prefixes) + ").*" if vision_prefixes else None

if CONFIG["lora"]["adapt_vision_tower"]:
    targets = sorted(set(targets) | (wanted & present_vis))
    EXCLUDE_REGEX = None
    print("Vision tower included. Final targets:", targets)
else:
    print("Vision tower frozen (adapt_vision_tower = False).")
    print("Exclusion regex:", EXCLUDE_REGEX or
          "none needed (no vision-named linear modules found)")

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

if USE_4BIT:
    base_model = prepare_model_for_kbit_training(
        base_model, use_gradient_checkpointing=CONFIG["train"]["gradient_checkpointing"])

lora_cfg = LoraConfig(
    r=CONFIG["lora"]["r"],
    lora_alpha=CONFIG["lora"]["alpha"],
    lora_dropout=CONFIG["lora"]["dropout"],
    bias=CONFIG["lora"]["bias"],
    target_modules=targets,
    task_type=TaskType.CAUSAL_LM,
    # Exclude the vision stack by name when we are not adapting it, so a shared leaf name such as
    # q_proj inside the ViT blocks cannot be picked up accidentally.
    exclude_modules=EXCLUDE_REGEX,
)
model = get_peft_model(base_model, lora_cfg)

if CONFIG["train"]["gradient_checkpointing"]:
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    if hasattr(model, "enable_input_require_grads"):
        model.enable_input_require_grads()
model.config.use_cache = False

# Keep the trainable LoRA parameters in fp32 for numerically stable updates under fp16 autocast.
n_cast = 0
for n, p in model.named_parameters():
    if p.requires_grad and p.dtype in (torch.float16, torch.bfloat16):
        p.data = p.data.float()
        n_cast += 1
print(f"LoRA parameters cast to fp32: {n_cast}")

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
pct = 100.0 * trainable_params / total_params

print("=" * 60)
print(f"Total parameters      : {total_params:,}")
print(f"Trainable parameters  : {trainable_params:,}")
print(f"Trainable percentage  : {pct:.4f} %")
print("=" * 60)
model.print_trainable_parameters()

PARAM_SUMMARY = {"total_parameters": int(total_params),
                 "trainable_parameters": int(trainable_params),
                 "trainable_percent": round(pct, 6),
                 "lora_targets": targets, "lora_r": CONFIG["lora"]["r"],
                 "lora_alpha": CONFIG["lora"]["alpha"], "load_in_4bit": USE_4BIT,
                 "compute_dtype": str(COMPUTE_DTYPE)}

### 10b. Experiment configuration summary

Printed before any expensive computation starts, so a run that is misconfigured can be killed in
the first minute rather than the third hour.

In [ ]:
def gpu_mem():
    if not torch.cuda.is_available():
        return "n/a"
    return (f"allocated {torch.cuda.memory_allocated()/1024**3:.2f} GiB / "
            f"reserved {torch.cuda.memory_reserved()/1024**3:.2f} GiB")

eff_bs = CONFIG["train"]["per_device_batch_size"] * CONFIG["train"]["grad_accum_steps"]
steps_per_epoch = math.ceil(len(train_df) / eff_bs)

print("=" * 72)
print("EXPERIMENT CONFIGURATION")
print("=" * 72)
rows = [
    ("Experiment", CONFIG["experiment_name"]),
    ("Model", MODEL_SOURCE),
    ("Precision", f"{COMPUTE_DTYPE}{' + 4-bit NF4' if USE_4BIT else ''}"),
    ("GPU", GPU_INFO[0]["name"] if GPU_INFO else "CPU"),
    ("VRAM", f"{TOTAL_VRAM:.1f} GiB"),
    ("Memory now", gpu_mem()),
    ("Classes", ", ".join(CLASSES)),
    ("Images (decodable)", len(CLEAN)),
    ("Train / Val / Test", f"{len(train_df)} / {len(val_df)} / {len(test_df)}"),
    ("Duplicate components", dup_report["distinct_groups"]),
    ("Micro batch", CONFIG["train"]["per_device_batch_size"]),
    ("Grad accumulation", CONFIG["train"]["grad_accum_steps"]),
    ("Effective batch", eff_bs),
    ("Optimiser steps/epoch", steps_per_epoch),
    ("Epochs", CONFIG["train"]["epochs"]),
    ("Learning rate", CONFIG["train"]["lr"]),
    ("Max pixels/image", f"{CONFIG['max_pixels']} (~{CONFIG['max_pixels']//(28*28*4)} visual tokens)"),
    ("Trainable params", f"{trainable_params:,} ({pct:.4f}%)"),
    ("Model selection", "validation macro-F1, early stopping patience "
                        f"{CONFIG['train']['early_stopping_patience']}"),
    ("Seed", SEED),
]
for k, v in rows:
    print(f"  {k:<22}: {v}")
print("=" * 72)

---
## 11. Training

A hand-written loop is used instead of `Trainer` because evaluation here is label scoring rather
than token generation, and the loop makes the masking, the accumulation and the model-selection
rule visible rather than hidden behind callbacks.

Memory measures in force: LoRA only, 4-bit base weights when VRAM is tight, gradient checkpointing,
micro-batch of 1 with accumulation, a capped visual-token budget, and `use_cache` disabled during
training.

**Model selection**: validation macro-F1 after each epoch. The best adapter is kept in memory and
written to disk. Early stopping triggers after
`early_stopping_patience` epochs without improvement. The test split is not read anywhere in this
section.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

@torch.no_grad()
def score_dataframe(df, desc="", batch_images=None, return_embeddings=False):
    """Return (probs [N,5], preds [N], row order) by scoring the five class labels per image.

    Two paths. When the five labels start with distinct tokens we take one forward pass and
    renormalise the next-token distribution over those five ids, which is exact for the first
    token. Otherwise each image is paired with all five candidate continuations and we sum the
    token log-probabilities of each full label string.
    """
    model.eval()
    bs = batch_images or CONFIG["train"]["eval_batch_images"]
    ds = MemeDataset(df, train=False, with_answer=False)
    dl = DataLoader(ds, batch_size=bs, shuffle=False, num_workers=2,
                    collate_fn=lambda b: collate(b, pad_side="left"), pin_memory=True)
    all_logprobs, all_emb, order = [], [], []
    ft = FIRST_TOKEN_TENSOR.to(DEVICE)
    t0 = time.time()
    for bi, batch in enumerate(dl):
        order.extend(batch["row_index"].tolist())
        inputs = {k: (v.to(DEVICE) if torch.is_tensor(v) else v)
                  for k, v in batch.items() if k not in ("row_index", "label_id", "labels")}
        with torch.autocast("cuda", dtype=COMPUTE_DTYPE, enabled=torch.cuda.is_available()):
            out = model(**inputs, use_cache=False, output_hidden_states=return_embeddings)
        if FAST_SCORING:
            logits = out.logits[:, -1, :].float()            # left padding -> last col is real
            lp = torch.log_softmax(logits, dim=-1)[:, ft]     # [B,5]
            lp = torch.log_softmax(lp, dim=-1)                # renormalise over the five classes
        else:
            lp = _score_full_labels(batch, inputs)
        all_logprobs.append(lp.float().cpu())
        if return_embeddings:
            h = out.hidden_states[-1].float()                 # [B,L,H]
            m = inputs["attention_mask"].unsqueeze(-1).float()
            all_emb.append(((h * m).sum(1) / m.sum(1).clamp(min=1)).cpu())
        del out, inputs
        if (bi + 1) % 25 == 0:
            torch.cuda.empty_cache()
            print(f"    {desc} {min((bi+1)*bs, len(ds))}/{len(ds)}  ({time.time()-t0:.0f}s)",
                  flush=True)
    logprobs = torch.cat(all_logprobs)
    probs = logprobs.exp().numpy()
    order = np.array(order)
    inv = np.argsort(order)
    probs = probs[inv]
    embs = torch.cat(all_emb).numpy()[inv] if return_embeddings else None
    preds = probs.argmax(1)
    return probs, preds, embs

@torch.no_grad()
def _score_full_labels(batch, inputs):
    """Fallback: sum token log-probs of every full label string. Five passes per batch."""
    B = inputs["input_ids"].shape[0]
    scores = torch.zeros(B, NUM_CLASSES, device=DEVICE)
    for ci, cls in enumerate(CLASSES):
        ids = torch.tensor(LABEL_TOKEN_IDS[cls], device=DEVICE).unsqueeze(0).expand(B, -1)
        ext = {**inputs,
               "input_ids": torch.cat([inputs["input_ids"], ids], dim=1),
               "attention_mask": torch.cat(
                   [inputs["attention_mask"], torch.ones_like(ids)], dim=1)}
        with torch.autocast("cuda", dtype=COMPUTE_DTYPE, enabled=torch.cuda.is_available()):
            out = model(**ext, use_cache=False)
        n = ids.shape[1]
        lg = torch.log_softmax(out.logits[:, -n-1:-1, :].float(), dim=-1)
        scores[:, ci] = lg.gather(2, ids.unsqueeze(-1)).squeeze(-1).sum(1)
        del out, ext
    return torch.log_softmax(scores, dim=-1)

def classification_metrics(y_true, y_pred):
    return {"accuracy": float(accuracy_score(y_true, y_pred)),
            "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
            "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0))}

In [ ]:
@torch.no_grad()
def teacher_forced_loss(df, max_samples=None):
    """Mean cross-entropy over answer tokens only. Comparable across train and validation."""
    model.eval()
    d = df if max_samples is None else df.head(max_samples)
    dl = DataLoader(MemeDataset(d, train=False, with_answer=True),
                    batch_size=CONFIG["train"]["per_device_batch_size"], shuffle=False,
                    num_workers=2, collate_fn=collate)
    tot, ntok = 0.0, 0
    for batch in dl:
        inputs = {k: v.to(DEVICE) for k, v in batch.items()
                  if k not in ("row_index", "label_id")}
        n = int((inputs["labels"] != -100).sum())
        with torch.autocast("cuda", dtype=COMPUTE_DTYPE, enabled=torch.cuda.is_available()):
            out = model(**inputs, use_cache=False)
        tot += float(out.loss) * n
        ntok += n
        del out, inputs
    return tot / max(ntok, 1)

In [ ]:
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup
import copy

TR = CONFIG["train"]
train_pool = train_df if TR["max_train_samples"] is None else \
    train_df.sample(n=min(TR["max_train_samples"], len(train_df)), random_state=SEED)
val_pool = val_df if TR["max_eval_samples"] is None else \
    val_df.sample(n=min(TR["max_eval_samples"], len(val_df)), random_state=SEED)

# Fixed train subsample used only for the training-accuracy curve (scoring the full train split
# every epoch would roughly double the run time and adds nothing to model selection).
n_train_probe = min(len(train_pool), 200)
train_probe = train_pool.sample(n=n_train_probe, random_state=SEED).reset_index(drop=True)
print(f"Training on {len(train_pool)} images | validating on {len(val_pool)} | "
      f"train-accuracy probe uses a fixed subsample of {n_train_probe}")

train_loader = DataLoader(MemeDataset(train_pool, train=True, with_answer=True),
                          batch_size=TR["per_device_batch_size"], shuffle=True, num_workers=2,
                          collate_fn=collate, pin_memory=True, drop_last=False)

steps_per_epoch = math.ceil(len(train_loader) / TR["grad_accum_steps"])
total_steps = steps_per_epoch * TR["epochs"]
optimizer = AdamW([p for p in model.parameters() if p.requires_grad],
                  lr=TR["lr"], weight_decay=TR["weight_decay"])
scheduler = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=int(total_steps * TR["warmup_ratio"]), num_training_steps=total_steps)
scaler = torch.amp.GradScaler("cuda", enabled=(COMPUTE_DTYPE == torch.float16 and
                                               torch.cuda.is_available()))
print(f"{steps_per_epoch} optimiser steps/epoch, {total_steps} total, "
      f"warmup {int(total_steps * TR['warmup_ratio'])}, GradScaler enabled: {scaler.is_enabled()}")

In [ ]:
history = {"step_loss": [], "epoch": []}
best = {"macro_f1": -1.0, "epoch": -1, "state": None}
patience_left = TR["early_stopping_patience"]
global_step = 0
set_seed(SEED)
t_start = time.time()

for epoch in range(1, TR["epochs"] + 1):
    model.train()
    model.config.use_cache = False
    running, seen, t_ep = 0.0, 0, time.time()
    optimizer.zero_grad(set_to_none=True)

    for it, batch in enumerate(train_loader):
        inputs = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()
                  if k not in ("row_index", "label_id")}
        with torch.autocast("cuda", dtype=COMPUTE_DTYPE, enabled=torch.cuda.is_available()):
            out = model(**inputs, use_cache=False)
            loss = out.loss / TR["grad_accum_steps"]
        scaler.scale(loss).backward()
        running += float(out.loss); seen += 1

        if (it + 1) % TR["grad_accum_steps"] == 0 or (it + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], TR["max_grad_norm"])
            scaler.step(optimizer); scaler.update()
            optimizer.zero_grad(set_to_none=True); scheduler.step()
            global_step += 1
            if global_step % TR["log_every_steps"] == 0:
                history["step_loss"].append(
                    {"step": global_step, "epoch": epoch, "loss": running / max(seen, 1),
                     "lr": scheduler.get_last_lr()[0]})
                print(f"  ep {epoch} step {global_step}/{total_steps} "
                      f"loss {running/max(seen,1):.4f} lr {scheduler.get_last_lr()[0]:.2e} "
                      f"| {gpu_mem()}", flush=True)
                running, seen = 0.0, 0
        del out, inputs, loss

    # ---- end-of-epoch evaluation (validation only) ----
    torch.cuda.empty_cache()
    print(f"  [epoch {epoch}] evaluating ...", flush=True)
    tr_loss = teacher_forced_loss(train_probe)
    va_loss = teacher_forced_loss(val_pool)
    _, tr_pred, _ = score_dataframe(train_probe, desc="train-probe")
    _, va_pred, _ = score_dataframe(val_pool, desc="val")
    tr_m = classification_metrics(train_probe["label_id"].values, tr_pred)
    va_m = classification_metrics(val_pool["label_id"].values, va_pred)

    rec = {"epoch": epoch, "train_loss": tr_loss, "val_loss": va_loss,
           "train_acc": tr_m["accuracy"], "val_acc": va_m["accuracy"],
           "train_macro_f1": tr_m["macro_f1"], "val_macro_f1": va_m["macro_f1"],
           "val_weighted_f1": va_m["weighted_f1"],
           "minutes": round((time.time() - t_ep) / 60, 2)}
    history["epoch"].append(rec)
    print(f"[epoch {epoch}] train_loss {tr_loss:.4f} | val_loss {va_loss:.4f} | "
          f"train_acc {tr_m['accuracy']:.4f} | val_acc {va_m['accuracy']:.4f} | "
          f"val_macroF1 {va_m['macro_f1']:.4f} | {rec['minutes']} min", flush=True)

    if va_m["macro_f1"] > best["macro_f1"]:
        best = {"macro_f1": va_m["macro_f1"], "epoch": epoch,
                "state": copy.deepcopy({k: v.detach().cpu()
                                        for k, v in model.state_dict().items()
                                        if "lora_" in k})}
        patience_left = TR["early_stopping_patience"]
        model.save_pretrained(OUT / "model" / "best_adapter")
        print(f"   new best validation macro-F1 {best['macro_f1']:.4f}; adapter saved")
    else:
        patience_left -= 1
        print(f"   no improvement; patience left {patience_left}")
        if patience_left <= 0:
            print("   early stopping")
            break

print(f"\nTraining finished in {(time.time()-t_start)/60:.1f} min. "
      f"Best epoch {best['epoch']} with validation macro-F1 {best['macro_f1']:.4f}")

In [ ]:
# ---- Restore the best checkpoint before any test-set contact ---------------------------------
if best["state"] is not None:
    missing_keys, unexpected = model.load_state_dict(
        {k: v.to(DEVICE) for k, v in best["state"].items()}, strict=False)
    print(f"Restored LoRA weights from epoch {best['epoch']} "
          f"({len(best['state'])} tensors; {len(unexpected)} unexpected keys)")
else:
    print("!! No checkpoint improved on the initial score; the final-epoch weights are in use.")

hist_df = pd.DataFrame(history["epoch"])
step_df = pd.DataFrame(history["step_loss"])
hist_df.to_csv(OUT / "metrics" / "training_history.csv", index=False)
step_df.to_csv(OUT / "metrics" / "step_loss.csv", index=False)
with open(OUT / "metrics" / "training_history.json", "w") as f:
    json.dump(history, f, indent=2)
print(hist_df.round(4).to_string(index=False))

---
## 12. Training Curves

Loss is the answer-token cross-entropy. Accuracy and macro-F1 come from the five-way scoring
procedure. The training accuracy line is measured on a fixed random subsample of the training split
(see Section 11), which is stated on the figure so it is not mistaken for the full-split value.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8.5))
ep = hist_df["epoch"].values

ax = axes[0, 0]
if len(step_df):
    ax.plot(step_df["step"], step_df["loss"], lw=1.2, color="#4C72B0")
    ax.set_xlabel("optimiser step"); ax.set_ylabel("loss")
    ax.set_title("Training loss per logged step")
else:
    ax.text(0.5, 0.5, "no step logs (run too short)", ha="center")

ax = axes[0, 1]
ax.plot(ep, hist_df["train_loss"], "o-", label="train", color="#4C72B0")
ax.plot(ep, hist_df["val_loss"], "s-", label="validation", color="#DD8452")
if best["epoch"] > 0:
    ax.axvline(best["epoch"], ls="--", c="gray", lw=1, label=f"selected epoch {best['epoch']}")
ax.set_xlabel("epoch"); ax.set_ylabel("answer-token cross-entropy")
ax.set_title("Train vs validation loss"); ax.legend(); ax.set_xticks(ep)

ax = axes[1, 0]
ax.plot(ep, hist_df["train_acc"], "o-", label=f"train (subsample n={n_train_probe})",
        color="#4C72B0")
ax.plot(ep, hist_df["val_acc"], "s-", label="validation", color="#DD8452")
ax.axhline(1.0 / NUM_CLASSES, ls=":", c="k", lw=1, label="random baseline (0.20)")
ax.set_xlabel("epoch"); ax.set_ylabel("accuracy"); ax.set_ylim(0, 1.02)
ax.set_title("Accuracy"); ax.legend(fontsize=8); ax.set_xticks(ep)

ax = axes[1, 1]
ax.plot(ep, hist_df["val_macro_f1"], "s-", color="#55A868", label="validation macro-F1")
ax.plot(ep, hist_df["val_weighted_f1"], "^-", color="#C44E52", label="validation weighted-F1")
if best["epoch"] > 0:
    ax.axvline(best["epoch"], ls="--", c="gray", lw=1)
    ax.scatter([best["epoch"]], [best["macro_f1"]], s=110, facecolors="none",
               edgecolors="black", zorder=5, label="model selection point")
ax.set_xlabel("epoch"); ax.set_ylabel("F1"); ax.set_ylim(0, 1.02)
ax.set_title("Validation F1"); ax.legend(fontsize=8); ax.set_xticks(ep)

fig.suptitle("Qwen-VL + LoRA training dynamics", y=0.995)
fig.tight_layout(); save_fig(fig, "12_training_curves"); plt.show()

---
## 13. Final Test Evaluation

The test split is opened here for the first time. Training and model selection are complete; the
weights are the ones chosen by validation macro-F1 in Section 11. This cell runs **once**. If you
change a hyper-parameter afterwards, the honest procedure is to re-run the whole notebook, not to
re-run this cell against a re-tuned model.

The forward pass also returns a mean-pooled final-layer hidden state per image, so the embeddings
used in Sections 15-17 come from the same single pass over the test data.

In [ ]:
print(f"Scoring the held-out test split: {len(test_df)} images")
t0 = time.time()
test_probs, test_preds, test_embs = score_dataframe(
    test_df, desc="test", return_embeddings=True)
print(f"Done in {(time.time()-t0)/60:.1f} min | probs {test_probs.shape} | "
      f"embeddings {None if test_embs is None else test_embs.shape}")

y_true = test_df["label_id"].values
y_pred = test_preds
test_conf = test_probs.max(axis=1)
assert len(y_true) == len(y_pred) == len(test_df)

In [ ]:
from sklearn.metrics import (classification_report, confusion_matrix,
                             precision_recall_fscore_support, balanced_accuracy_score,
                             cohen_kappa_score, matthews_corrcoef)

acc = accuracy_score(y_true, y_pred)
bal_acc = balanced_accuracy_score(y_true, y_pred)
macro_p, macro_r, macro_f, _ = precision_recall_fscore_support(
    y_true, y_pred, average="macro", zero_division=0, labels=range(NUM_CLASSES))
w_p, w_r, w_f, _ = precision_recall_fscore_support(
    y_true, y_pred, average="weighted", zero_division=0, labels=range(NUM_CLASSES))
micro_f = f1_score(y_true, y_pred, average="micro", zero_division=0)
kappa = cohen_kappa_score(y_true, y_pred)
mcc = matthews_corrcoef(y_true, y_pred)

p, r, f1, sup = precision_recall_fscore_support(
    y_true, y_pred, labels=range(NUM_CLASSES), zero_division=0)
per_class = pd.DataFrame({"Class": CLASSES, "Precision": p.round(4), "Recall": r.round(4),
                          "F1": f1.round(4), "Support": sup.astype(int)})

print("=" * 78)
print("TEST-SET RESULTS  (held out, evaluated once)")
print("=" * 78)
print(per_class.to_string(index=False))
print("-" * 78)
print(f"Overall Accuracy   : {acc:.4f}")
print(f"Balanced Accuracy  : {bal_acc:.4f}")
print(f"Macro Precision    : {macro_p:.4f}")
print(f"Macro Recall       : {macro_r:.4f}")
print(f"Macro F1           : {macro_f:.4f}")
print(f"Weighted Precision : {w_p:.4f}")
print(f"Weighted Recall    : {w_r:.4f}")
print(f"Weighted F1        : {w_f:.4f}")
print(f"Micro F1           : {micro_f:.4f}")
print(f"Cohen kappa        : {kappa:.4f}")
print(f"Matthews corrcoef  : {mcc:.4f}")
print(f"Random baseline acc: {1/NUM_CLASSES:.4f} | majority-class baseline: "
      f"{test_df['class'].value_counts().max()/len(test_df):.4f}")
print("=" * 78)

TEST_METRICS = {
    "n_test": int(len(test_df)), "accuracy": float(acc), "balanced_accuracy": float(bal_acc),
    "macro_precision": float(macro_p), "macro_recall": float(macro_r), "macro_f1": float(macro_f),
    "weighted_precision": float(w_p), "weighted_recall": float(w_r), "weighted_f1": float(w_f),
    "micro_f1": float(micro_f), "cohen_kappa": float(kappa), "mcc": float(mcc),
    "random_baseline_accuracy": 1.0 / NUM_CLASSES,
    "majority_baseline_accuracy": float(test_df["class"].value_counts().max() / len(test_df)),
    "selected_epoch": int(best["epoch"]), "selection_val_macro_f1": float(best["macro_f1"]),
}
per_class.to_csv(OUT / "metrics" / "classification_report.csv", index=False)
print(classification_report(y_true, y_pred, target_names=CLASSES, digits=4, zero_division=0))

In [ ]:
# ---- Prediction table --------------------------------------------------------------------------
pred_df = test_df[["filepath", "filename", "class", "label_id", "width", "height",
                   "aspect_ratio", "file_size_kb", "dup_group"]].copy()
pred_df["true_class"] = pred_df["class"]
pred_df["pred_label_id"] = y_pred
pred_df["pred_class"] = [ID_TO_CLASS[i] for i in y_pred]
pred_df["correct"] = pred_df["true_class"] == pred_df["pred_class"]
pred_df["confidence"] = test_conf
for i, c in enumerate(CLASSES):
    pred_df[f"p_{c}"] = test_probs[:, i]
pred_df["margin"] = np.sort(test_probs, axis=1)[:, -1] - np.sort(test_probs, axis=1)[:, -2]
pred_df["entropy"] = -(test_probs * np.log(np.clip(test_probs, 1e-12, 1))).sum(1)
pred_df.drop(columns=["class"]).to_csv(OUT / "predictions" / "test_predictions.csv", index=False)
print(f"{int(pred_df['correct'].sum())}/{len(pred_df)} correct "
      f"({pred_df['correct'].mean()*100:.2f}%)")
print("\nMean confidence when correct  :", round(pred_df.loc[pred_df['correct'], 'confidence'].mean(), 4))
print("Mean confidence when incorrect:",
      round(pred_df.loc[~pred_df['correct'], 'confidence'].mean(), 4)
      if (~pred_df['correct']).any() else "n/a (no errors)")

---
## 14. Confusion Matrix

Rows are the true class, columns the predicted class, both in the fixed order
Political, Religious, Sports, Educational, Harmless. Raw counts and row-normalised rates are shown
side by side, and the dominant confusions are then extracted programmatically rather than read off
the figure by eye.

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=range(NUM_CLASSES))
cm_norm = cm.astype(float) / np.clip(cm.sum(axis=1, keepdims=True), 1, None)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", square=True, cbar_kws={"shrink": 0.8},
            xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[0], linewidths=0.5, linecolor="white")
axes[0].set_title(f"Confusion matrix, raw counts (n = {len(y_true)})")
axes[0].set_xlabel("Predicted class"); axes[0].set_ylabel("True class")

sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues", square=True, vmin=0, vmax=1,
            cbar_kws={"shrink": 0.8}, xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[1],
            linewidths=0.5, linecolor="white")
axes[1].set_title("Row-normalised (per-true-class recall on the diagonal)")
axes[1].set_xlabel("Predicted class"); axes[1].set_ylabel("True class")
for ax in axes:
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
fig.tight_layout(); save_fig(fig, "14_confusion_matrix"); plt.show()

np.savetxt(OUT / "metrics" / "confusion_matrix_counts.csv", cm, fmt="%d", delimiter=",",
           header=",".join(CLASSES), comments="")
np.savetxt(OUT / "metrics" / "confusion_matrix_normalized.csv", cm_norm, fmt="%.6f", delimiter=",",
           header=",".join(CLASSES), comments="")

In [ ]:
# ---- Programmatic reading of the confusion structure ------------------------------------------
off = cm.copy(); np.fill_diagonal(off, 0)
off_rate = cm_norm.copy(); np.fill_diagonal(off_rate, 0)

confusions = []
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        if i != j and cm[i, j] > 0:
            confusions.append({"true": CLASSES[i], "predicted": CLASSES[j],
                               "count": int(cm[i, j]), "rate_of_true_class": float(cm_norm[i, j])})
conf_df = pd.DataFrame(confusions).sort_values("count", ascending=False)

recall_per_class = np.diag(cm_norm)
diag = pd.DataFrame({"Class": CLASSES, "Recall": recall_per_class.round(4),
                     "Precision": p.round(4), "F1": f1.round(4), "Support": sup})

print("DIRECTED CONFUSIONS (true -> predicted), most frequent first")
print(conf_df.head(12).to_string(index=False) if len(conf_df) else "  none: no test error at all")

if len(conf_df):
    top = conf_df.iloc[0]
    print(f"\nHighest-count confusion direction: {top['true']} -> {top['predicted']} "
          f"({top['count']} images, {top['rate_of_true_class']*100:.1f}% of all "
          f"{top['true']} test images)")

    # Symmetric pair confusion: how entangled are two classes in both directions
    pair_rows = []
    for i in range(NUM_CLASSES):
        for j in range(i + 1, NUM_CLASSES):
            tot = int(cm[i, j] + cm[j, i])
            if tot:
                pair_rows.append({"class_a": CLASSES[i], "class_b": CLASSES[j],
                                  "a_to_b": int(cm[i, j]), "b_to_a": int(cm[j, i]),
                                  "total": tot,
                                  "symmetric_rate": float((cm_norm[i, j] + cm_norm[j, i]) / 2)})
    pair_df = pd.DataFrame(pair_rows).sort_values("total", ascending=False)
    print("\nMOST CONFUSED CLASS PAIRS (both directions combined)")
    print(pair_df.to_string(index=False))
    MOST_CONFUSED_PAIR = f"{pair_df.iloc[0]['class_a']} <-> {pair_df.iloc[0]['class_b']}"
    TOP_DIRECTION = f"{top['true']} -> {top['predicted']}"
else:
    pair_df = pd.DataFrame()
    MOST_CONFUSED_PAIR = "none (no misclassifications)"
    TOP_DIRECTION = "none"

best_i, worst_i = int(np.argmax(f1)), int(np.argmin(f1))
BEST_CLASS, WORST_CLASS = CLASSES[best_i], CLASSES[worst_i]
print(f"\nStrongest class by F1 : {BEST_CLASS} (F1 {f1[best_i]:.4f}, recall {recall_per_class[best_i]:.4f})")
print(f"Weakest class by F1   : {WORST_CLASS} (F1 {f1[worst_i]:.4f}, recall {recall_per_class[worst_i]:.4f})")
print(f"Most confused pair    : {MOST_CONFUSED_PAIR}")
print(f"Strongest direction   : {TOP_DIRECTION}")
if len(conf_df):
    conf_df.to_csv(OUT / "metrics" / "confusion_directions.csv", index=False)
    pair_df.to_csv(OUT / "metrics" / "confused_pairs.csv", index=False)

---
## 15. Feature Extraction

The representation used for the geometric analyses is the **mean-pooled final-layer hidden state of
the fine-tuned model over the full prompt sequence**, i.e. the vector the language head reads from
just before it commits to a class token. It therefore reflects both the visual evidence and the
instruction context.

Two caveats to state in the thesis:

1. This is a decoder representation of an instruction-conditioned sequence, not a pure image
   embedding. Two memes can be close here because the model plans the same answer for them.
2. Because the model was fine-tuned for this label set, the space is biased towards separating these
   five classes. It is a view of what the trained model has learned, not of Bangladeshi meme
   semantics in general.

Test embeddings come from the single test pass in Section 13. Validation embeddings are extracted
here as well, so the geometry can be inspected on data that never contributed a gradient and was
not the reporting set.

In [ ]:
print("Test embeddings:", test_embs.shape)
print(f"Extracting validation embeddings ({len(val_pool)} images) ...")
val_probs, val_preds, val_embs = score_dataframe(val_pool, desc="val-emb", return_embeddings=True)
print("Validation embeddings:", val_embs.shape)

EMB = test_embs.astype(np.float32)
EMB_LABELS = y_true
EMB_NORM = EMB / np.clip(np.linalg.norm(EMB, axis=1, keepdims=True), 1e-12, None)

np.save(OUT / "embeddings" / "test_embeddings.npy", EMB)
np.save(OUT / "embeddings" / "test_labels.npy", EMB_LABELS)
np.save(OUT / "embeddings" / "test_pred_labels.npy", y_pred)
np.save(OUT / "embeddings" / "val_embeddings.npy", val_embs.astype(np.float32))
np.save(OUT / "embeddings" / "val_labels.npy", val_pool["label_id"].values)
print("Embedding dimensionality:", EMB.shape[1])
print("Saved to", OUT / "embeddings")

---
## 16. t-SNE / UMAP Analysis

**Read this before interpreting the plots.** t-SNE and UMAP are non-linear, stochastic embeddings
that preserve local neighbourhoods only. Cluster sizes, inter-cluster distances and empty space in
these plots are not meaningful, and a visual overlap is not proof that two classes are semantically
inseparable. They are used here as an exploratory aid. The quantitative separability numbers in the
next cell are computed in the **original high-dimensional space**, which is where the claims belong.

In [ ]:
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

n_emb = len(EMB)
perplexity = int(max(5, min(30, (n_emb - 1) / 3)))
pca_dim = int(min(50, EMB.shape[1], max(2, n_emb - 1)))
pca_emb = PCA(n_components=pca_dim, random_state=SEED).fit_transform(EMB)
print(f"PCA {EMB.shape[1]} -> {pca_dim} dims before t-SNE; perplexity {perplexity}")

tsne_xy = TSNE(n_components=2, perplexity=perplexity, init="pca", learning_rate="auto",
               random_state=SEED, max_iter=1000).fit_transform(pca_emb)

umap_xy, UMAP_OK = None, False
try:
    import umap
    umap_xy = umap.UMAP(n_components=2, n_neighbors=int(min(15, max(2, n_emb - 1))),
                        min_dist=0.1, metric="cosine",
                        random_state=SEED).fit_transform(EMB)
    UMAP_OK = True
except Exception as e:
    print("UMAP unavailable, showing t-SNE only:", type(e).__name__, e)

ncols = 2 if UMAP_OK else 1
fig, axes = plt.subplots(1, ncols, figsize=(8 * ncols, 7), squeeze=False)
for ax, (xy, name) in zip(axes[0], [(tsne_xy, "t-SNE"), (umap_xy, "UMAP")][:ncols]):
    for ci, cls in enumerate(CLASSES):
        m = EMB_LABELS == ci
        ax.scatter(xy[m, 0], xy[m, 1], s=26, alpha=0.75, label=f"{cls} (n={int(m.sum())})",
                   color=CLASS_COLORS[cls], edgecolors="white", linewidths=0.4)
    ax.set_title(f"{name} of fine-tuned Qwen-VL features (test split)")
    ax.set_xlabel(f"{name}-1"); ax.set_ylabel(f"{name}-2")
    ax.legend(fontsize=8, loc="best")
fig.suptitle("Exploratory projection only. Distances and cluster sizes are not interpretable.",
             y=0.995, fontsize=9, style="italic")
fig.tight_layout(); save_fig(fig, "16_tsne_umap"); plt.show()

np.save(OUT / "embeddings" / "tsne_xy.npy", tsne_xy)
if UMAP_OK:
    np.save(OUT / "embeddings" / "umap_xy.npy", umap_xy)

In [ ]:
# ---- Correct vs incorrect overlay --------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 7))
ok_mask = pred_df["correct"].values
for ci, cls in enumerate(CLASSES):
    m = (EMB_LABELS == ci) & ok_mask
    ax.scatter(tsne_xy[m, 0], tsne_xy[m, 1], s=24, alpha=0.6, color=CLASS_COLORS[cls], label=cls)
m = ~ok_mask
ax.scatter(tsne_xy[m, 0], tsne_xy[m, 1], s=70, marker="X", facecolors="none",
           edgecolors="black", linewidths=1.2, label=f"misclassified (n={int(m.sum())})")
ax.set_title("t-SNE with misclassified test images marked")
ax.legend(fontsize=8)
fig.tight_layout(); save_fig(fig, "16b_tsne_errors"); plt.show()

In [ ]:
# ---- Quantitative separability, computed in the ORIGINAL feature space -------------------------
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.neighbors import NearestNeighbors

sil_overall = float(silhouette_score(EMB_NORM, EMB_LABELS, metric="cosine")) \
    if len(np.unique(EMB_LABELS)) > 1 else float("nan")
sil_per = silhouette_samples(EMB_NORM, EMB_LABELS, metric="cosine")

K = int(min(10, max(1, len(EMB) - 1)))
nn = NearestNeighbors(n_neighbors=K + 1, metric="cosine").fit(EMB_NORM)
_, idx = nn.kneighbors(EMB_NORM)
neigh_labels = EMB_LABELS[idx[:, 1:]]
knn_purity = (neigh_labels == EMB_LABELS[:, None]).mean(axis=1)

sep = pd.DataFrame({
    "Class": CLASSES,
    "n_test": [int((EMB_LABELS == i).sum()) for i in range(NUM_CLASSES)],
    "silhouette": [round(float(sil_per[EMB_LABELS == i].mean()), 4) if (EMB_LABELS == i).any()
                   else np.nan for i in range(NUM_CLASSES)],
    f"knn{K}_purity": [round(float(knn_purity[EMB_LABELS == i].mean()), 4) if (EMB_LABELS == i).any()
                       else np.nan for i in range(NUM_CLASSES)],
})
print(f"Overall cosine silhouette (test, original {EMB.shape[1]}-d space): {sil_overall:.4f}")
print("  Silhouette runs from -1 to 1. Values near 0 mean class regions touch; negative means the "
      "average sample sits closer to another class than to its own.")
print(f"\nPer-class separability (k = {K} neighbours):")
print(sep.to_string(index=False))

# Which class do the wrong-class neighbours belong to?
print("\nNearest-neighbour label leakage, share of each class's k neighbours by class:")
leak = np.zeros((NUM_CLASSES, NUM_CLASSES))
for i in range(NUM_CLASSES):
    m = EMB_LABELS == i
    if m.any():
        vals, cnts = np.unique(neigh_labels[m], return_counts=True)
        leak[i, vals] = cnts / cnts.sum()
leak_df = pd.DataFrame(leak.round(3), index=CLASSES, columns=CLASSES)
print(leak_df.to_string())
sep.to_csv(OUT / "metrics" / "separability.csv", index=False)
leak_df.to_csv(OUT / "metrics" / "neighbour_label_shares.csv")

---
## 17. Class Similarity Analysis

Each class centroid is the mean of the L2-normalised embeddings of its test images. The 5x5 matrix
below is the cosine similarity between those centroids.

Interpretation boundary: this measures how similarly the **fine-tuned model represents** the two
classes at the point of decision. It is not a measure of human semantic similarity, and a high value
does not by itself imply the two categories are conceptually close. Whether a high similarity
coincides with actual confusion is checked directly against the confusion matrix at the end of the
cell.

In [ ]:
centroids = np.stack([EMB_NORM[EMB_LABELS == i].mean(axis=0) if (EMB_LABELS == i).any()
                      else np.full(EMB_NORM.shape[1], np.nan) for i in range(NUM_CLASSES)])
centroids_n = centroids / np.clip(np.linalg.norm(centroids, axis=1, keepdims=True), 1e-12, None)
sim = centroids_n @ centroids_n.T
sim_df = pd.DataFrame(sim, index=CLASSES, columns=CLASSES)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sns.heatmap(sim_df, annot=True, fmt=".3f", cmap="RdYlBu_r", square=True, vmin=sim.min(), vmax=1.0,
            linewidths=0.5, linecolor="white", ax=axes[0], cbar_kws={"shrink": 0.8})
axes[0].set_title("Cosine similarity between class centroids (test embeddings)")

mask = np.triu(np.ones_like(sim, dtype=bool))
off_sim = np.where(mask, np.nan, sim)
sns.heatmap(pd.DataFrame(off_sim, index=CLASSES, columns=CLASSES), annot=True, fmt=".3f",
            cmap="RdYlBu_r", square=True, linewidths=0.5, linecolor="white", ax=axes[1],
            cbar_kws={"shrink": 0.8})
axes[1].set_title("Off-diagonal pairs only")
for ax in axes:
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
fig.tight_layout(); save_fig(fig, "17_class_similarity"); plt.show()
sim_df.round(6).to_csv(OUT / "metrics" / "class_centroid_similarity.csv")

pairs = [(CLASSES[i], CLASSES[j], float(sim[i, j]))
         for i in range(NUM_CLASSES) for j in range(i + 1, NUM_CLASSES)]
pairs_sorted = sorted(pairs, key=lambda x: -x[2])
print("Centroid-similarity ranking (most to least similar):")
for a, b, v in pairs_sorted:
    print(f"  {a:<12} <-> {b:<12} {v:.4f}")
MOST_SIMILAR_PAIR = f"{pairs_sorted[0][0]} <-> {pairs_sorted[0][1]} ({pairs_sorted[0][2]:.4f})"
LEAST_SIMILAR_PAIR = f"{pairs_sorted[-1][0]} <-> {pairs_sorted[-1][1]} ({pairs_sorted[-1][2]:.4f})"
print(f"\nMost similar pair  : {MOST_SIMILAR_PAIR}")
print(f"Least similar pair : {LEAST_SIMILAR_PAIR}")

# Does representation-space proximity line up with actual errors?
if len(pair_df):
    joined = []
    for a, b, v in pairs:
        row = pair_df[((pair_df["class_a"] == a) & (pair_df["class_b"] == b)) |
                      ((pair_df["class_a"] == b) & (pair_df["class_b"] == a))]
        joined.append({"pair": f"{a} <-> {b}", "centroid_similarity": round(v, 4),
                       "misclassified_both_ways": int(row["total"].iloc[0]) if len(row) else 0})
    jdf = pd.DataFrame(joined).sort_values("centroid_similarity", ascending=False)
    print("\nCentroid similarity next to observed confusion count:")
    print(jdf.to_string(index=False))
    if jdf["misclassified_both_ways"].std() > 0 and jdf["centroid_similarity"].std() > 0:
        rho = jdf["centroid_similarity"].corr(jdf["misclassified_both_ways"], method="spearman")
        print(f"\nSpearman correlation over the {len(jdf)} class pairs: rho = {rho:.3f}")
        print("  Descriptive only. With 10 pairs this is not a hypothesis test, and it says "
              "nothing about causation in either direction.")
    jdf.to_csv(OUT / "metrics" / "similarity_vs_confusion.csv", index=False)

---
## 18. Misclassification Analysis

Every incorrect test prediction is collected with its image, true label, predicted label and the
model's probability for both. Error groups are then displayed **only for the true -> predicted
directions that actually occurred**; nothing is assumed in advance about which confusions exist.

In [ ]:
errors = pred_df[~pred_df["correct"]].copy()
errors["p_true"] = [errors.iloc[i][f"p_{errors.iloc[i]['true_class']}"] for i in range(len(errors))]
errors["direction"] = errors["true_class"] + " -> " + errors["pred_class"]
errors = errors.sort_values("confidence", ascending=False).reset_index(drop=True)
errors.drop(columns=["class"], errors="ignore").to_csv(
    OUT / "predictions" / "misclassified.csv", index=False)

print(f"Total misclassified test images: {len(errors)} of {len(pred_df)} "
      f"({len(errors)/len(pred_df)*100:.2f}%)")
if len(errors):
    dir_counts = errors["direction"].value_counts()
    print("\nError directions observed:")
    print(dir_counts.to_string())
    print("\nError rate by true class:")
    er = (errors.groupby("true_class").size().reindex(CLASSES).fillna(0).astype(int)
          / pred_df.groupby("true_class").size().reindex(CLASSES)).round(4)
    print(er.to_string())
else:
    dir_counts = pd.Series(dtype=int)
    print("No misclassifications on the test split, so Sections 18 and 19 have nothing to show.")

In [ ]:
# ---- Representative examples per observed error direction ---------------------------------------
MAX_DIRECTIONS, PER_DIRECTION = 6, 4
if len(errors):
    top_dirs = dir_counts.head(MAX_DIRECTIONS).index.tolist()
    fig, axes = plt.subplots(len(top_dirs), PER_DIRECTION,
                             figsize=(3.1 * PER_DIRECTION, 3.4 * len(top_dirs)), squeeze=False)
    for r, d in enumerate(top_dirs):
        sub = errors[errors["direction"] == d].head(PER_DIRECTION)
        for c in range(PER_DIRECTION):
            ax = axes[r][c]; ax.axis("off")
            if c < len(sub):
                row = sub.iloc[c]
                try:
                    ax.imshow(Image.open(row["filepath"]).convert("RGB"))
                except Exception:
                    ax.text(0.5, 0.5, "unreadable", ha="center")
                ax.set_title(f"p({row['pred_class']})={row['confidence']:.2f}\n"
                             f"p({row['true_class']})={row['p_true']:.2f}", fontsize=8)
            if c == 0:
                ax.text(-0.06, 0.5, f"{d}\n(n={int(dir_counts[d])})", transform=ax.transAxes,
                        rotation=90, va="center", ha="center", fontsize=10, fontweight="bold")
    fig.suptitle("Representative misclassifications, grouped by the error directions that occurred",
                 y=1.0)
    fig.tight_layout(); save_fig(fig, "18_misclassifications"); plt.show()
else:
    print("Skipped: no errors to display.")

In [ ]:
# ---- Are errors associated with any measurable image property? ----------------------------------
if len(errors):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, col, lab in zip(axes, ["confidence", "entropy", "megapixels"],
                            ["max class probability", "predictive entropy (nats)", "megapixels"]):
        data = [pred_df.loc[pred_df["correct"], col], pred_df.loc[~pred_df["correct"], col]]
        ax.hist(data, bins=20, label=["correct", "incorrect"], color=["#55A868", "#C44E52"])
        ax.set_xlabel(lab); ax.set_ylabel("count"); ax.legend(fontsize=8)
    fig.suptitle("Measured properties of correct vs incorrect predictions")
    fig.tight_layout(); save_fig(fig, "18b_error_properties"); plt.show()

    comp = pd.DataFrame({
        "metric": ["mean confidence", "mean entropy", "mean megapixels", "mean aspect ratio",
                   "mean file size KB"],
        "correct": [pred_df.loc[pred_df["correct"], c].mean() for c in
                    ["confidence", "entropy", "megapixels", "aspect_ratio", "file_size_kb"]],
        "incorrect": [pred_df.loc[~pred_df["correct"], c].mean() for c in
                      ["confidence", "entropy", "megapixels", "aspect_ratio", "file_size_kb"]],
    }).round(4)
    print(comp.to_string(index=False))
    print("\nThese are group means on a small sample. They describe the errors; they do not "
          "establish that any property causes them.")

---
## 19. High-Confidence Wrong Predictions

The errors the model was most certain about. These are the most informative failures for a thesis,
because a confidently wrong prediction points at a systematic pattern rather than at noise.

The commentary printed underneath is restricted to what is actually measurable: the probability the
model assigned to each class, the runner-up class, the image geometry, and whether the file belongs
to a duplicate component. Any statement about *why* a specific meme was confused would require
reading the image, so that judgement is left to you; the cell prints a structured prompt for
recording it.

In [ ]:
TOP_K = min(8, len(errors))
if TOP_K:
    hc = errors.head(TOP_K)
    ncol = 4; nrow = math.ceil(TOP_K / ncol)
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.4 * ncol, 4.0 * nrow), squeeze=False)
    for k in range(nrow * ncol):
        ax = axes[k // ncol][k % ncol]; ax.axis("off")
        if k < TOP_K:
            row = hc.iloc[k]
            try:
                ax.imshow(Image.open(row["filepath"]).convert("RGB"))
            except Exception:
                ax.text(0.5, 0.5, "unreadable", ha="center")
            ax.set_title(f"true: {row['true_class']}\npred: {row['pred_class']} "
                         f"({row['confidence']:.3f})\np(true) = {row['p_true']:.3f}",
                         fontsize=9, color="#C44E52")
    fig.suptitle(f"The {TOP_K} most confident incorrect predictions", y=1.0)
    fig.tight_layout(); save_fig(fig, "19_high_confidence_errors"); plt.show()

    print("Measured detail for each high-confidence error:\n")
    for k in range(TOP_K):
        row = hc.iloc[k]
        dist_str = ", ".join(f"{c} {row[f'p_{c}']:.3f}" for c in CLASSES)
        others = sorted(((row[f"p_{c}"], c) for c in CLASSES if c != row["pred_class"]),
                        reverse=True)
        grp = int((CLEAN["dup_group"] == row["dup_group"]).sum())
        print(f"[{k+1}] {row['filename']}")
        print(f"     true {row['true_class']} | predicted {row['pred_class']} "
              f"| confidence {row['confidence']:.3f} | margin {row['margin']:.3f}")
        print(f"     full distribution: {dist_str}")
        print(f"     runner-up: {others[0][1]} ({others[0][0]:.3f}) | "
              f"true class ranked #{int(np.sum([row[f'p_{c}'] > row['p_true'] for c in CLASSES]) + 1)} of 5")
        print(f"     image {int(row['width'])}x{int(row['height'])} px, "
              f"{row['file_size_kb']:.0f} KB, duplicate-component size {grp}")
        print("     observation to record after viewing the image above: ______\n")
    hc.drop(columns=["class"], errors="ignore").to_csv(
        OUT / "predictions" / "high_confidence_errors.csv", index=False)
else:
    print("No misclassifications, so there are no high-confidence errors to inspect.")

---
## 20. Per-Class Analysis

One row per class combining the classification metrics, the dominant confusion direction taken from
the confusion matrix, and the feature-space behaviour from Sections 16-17. This is the table that
answers "why is one class harder than another", by putting the evidence for each class side by side.

In [ ]:
rows = []
for i, cls in enumerate(CLASSES):
    row_conf = cm[i].copy(); row_conf[i] = 0
    if row_conf.sum() > 0:
        j = int(np.argmax(row_conf))
        tendency = f"{CLASSES[j]} ({int(row_conf[j])} of {int(cm[i].sum())})"
    else:
        tendency = "none"
    col_conf = cm[:, i].copy(); col_conf[i] = 0
    attracts = (f"{CLASSES[int(np.argmax(col_conf))]} ({int(col_conf.max())})"
                if col_conf.sum() > 0 else "none")
    sims = [(sim[i, j], CLASSES[j]) for j in range(NUM_CLASSES) if j != i]
    nearest = max(sims)
    rows.append({
        "Class": cls,
        "Train": int((train_df["class"] == cls).sum()),
        "Val": int((val_df["class"] == cls).sum()),
        "Test support": int(sup[i]),
        "Precision": round(float(p[i]), 4),
        "Recall": round(float(r[i]), 4),
        "F1": round(float(f1[i]), 4),
        "Confused into": tendency,
        "Attracts from": attracts,
        "Silhouette": sep.loc[i, "silhouette"],
        "kNN purity": sep.loc[i, f"knn{K}_purity"],
        "Nearest centroid": f"{nearest[1]} ({nearest[0]:.3f})",
    })
per_class_full = pd.DataFrame(rows)
print(per_class_full.to_string(index=False))
per_class_full.to_csv(OUT / "metrics" / "per_class_analysis.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 4.6))
x = np.arange(NUM_CLASSES); w = 0.26
axes[0].bar(x - w, p, w, label="precision", color="#4C72B0")
axes[0].bar(x, r, w, label="recall", color="#DD8452")
axes[0].bar(x + w, f1, w, label="F1", color="#55A868")
axes[0].set_xticks(x); axes[0].set_xticklabels(CLASSES, rotation=20)
axes[0].set_ylim(0, 1.05); axes[0].set_title("Per-class test metrics"); axes[0].legend()

axes[1].scatter(per_class_full["Train"], per_class_full["F1"], s=110,
                c=[CLASS_COLORS[c] for c in CLASSES], edgecolors="black")
for i, cls in enumerate(CLASSES):
    axes[1].annotate(cls, (per_class_full["Train"][i], per_class_full["F1"][i]),
                     textcoords="offset points", xytext=(6, 5), fontsize=9)
axes[1].set_xlabel("training images for the class"); axes[1].set_ylabel("test F1")
axes[1].set_title("Class F1 against training-set size (descriptive, n = 5 points)")
fig.tight_layout(); save_fig(fig, "20_per_class"); plt.show()

print("\nReading guide, tied to the numbers above rather than to intuition:")
print(f"  - Strongest class: {BEST_CLASS} (F1 {f1[best_i]:.4f}).")
print(f"  - Weakest class  : {WORST_CLASS} (F1 {f1[worst_i]:.4f}).")
print("  - A class with low recall loses its own images to the 'Confused into' column; a class "
      "with low precision absorbs images listed in 'Attracts from'.")
print("  - Low silhouette and low kNN purity for the same class indicate its test embeddings sit "
      "inside another class's region, which is the geometric counterpart of that confusion.")

---
## 21. Final Results and Artifact Export

In [ ]:
SUMMARY = {
    "model": f"Qwen3-VL + LoRA ({MODEL_SOURCE})",
    "dataset": "Bangladeshi Meme Dataset (folder-organised, discovered at runtime)",
    "num_classes": NUM_CLASSES,
    "classes": CLASSES,
    "images_discovered": int(len(index_df)),
    "images_unreadable": int(len(bad)),
    "images_used": int(len(CLEAN)),
    "train_samples": int(len(train_df)),
    "val_samples": int(len(val_df)),
    "test_samples": int(len(test_df)),
    "split_ratios": ratios,
    "duplicates": dup_report,
    "parameters": PARAM_SUMMARY,
    "training": {k: v for k, v in CONFIG["train"].items()},
    "epochs_run": int(len(hist_df)),
    "selected_epoch": int(best["epoch"]),
    "selection_val_macro_f1": float(best["macro_f1"]),
    "test_metrics": TEST_METRICS,
    "best_class": f"{BEST_CLASS} (F1 {f1[best_i]:.4f})",
    "weakest_class": f"{WORST_CLASS} (F1 {f1[worst_i]:.4f})",
    "most_confused_pair": MOST_CONFUSED_PAIR,
    "strongest_confusion_direction": TOP_DIRECTION,
    "most_similar_centroid_pair": MOST_SIMILAR_PAIR,
    "least_similar_centroid_pair": LEAST_SIMILAR_PAIR,
    "embedding_silhouette_cosine": sil_overall,
    "environment": ENV_INFO,
    "seed": SEED,
    "prompt": CLASSIFICATION_PROMPT,
    "scoring_strategy": "first-token restricted softmax" if FAST_SCORING
                        else "summed label log-probabilities",
}

print("=" * 78)
print("EXPERIMENT SUMMARY")
print("=" * 78)
print(f"Model                : {SUMMARY['model']}")
print(f"Dataset              : {SUMMARY['dataset']}")
print(f"Number of Classes    : {NUM_CLASSES}")
print(f"Train Samples        : {SUMMARY['train_samples']}")
print(f"Validation Samples   : {SUMMARY['val_samples']}")
print(f"Test Samples         : {SUMMARY['test_samples']}")
print(f"Trainable Parameters : {PARAM_SUMMARY['trainable_parameters']:,} "
      f"({PARAM_SUMMARY['trainable_percent']:.4f}% of {PARAM_SUMMARY['total_parameters']:,})")
print(f"Epochs Run           : {SUMMARY['epochs_run']} (selected epoch {SUMMARY['selected_epoch']})")
print("-" * 78)
print(f"Accuracy             : {TEST_METRICS['accuracy']:.4f}")
print(f"Balanced Accuracy    : {TEST_METRICS['balanced_accuracy']:.4f}")
print(f"Macro-F1             : {TEST_METRICS['macro_f1']:.4f}")
print(f"Weighted-F1          : {TEST_METRICS['weighted_f1']:.4f}")
print(f"Cohen kappa          : {TEST_METRICS['cohen_kappa']:.4f}")
print(f"Random baseline      : {TEST_METRICS['random_baseline_accuracy']:.4f}")
print(f"Majority baseline    : {TEST_METRICS['majority_baseline_accuracy']:.4f}")
print("-" * 78)
print(f"Best Class           : {SUMMARY['best_class']}")
print(f"Weakest Class        : {SUMMARY['weakest_class']}")
print(f"Most Confused Pair   : {SUMMARY['most_confused_pair']}")
print(f"Strongest Direction  : {SUMMARY['strongest_confusion_direction']}")
print(f"Most Similar Pair    : {SUMMARY['most_similar_centroid_pair']}")
print(f"Least Similar Pair   : {SUMMARY['least_similar_centroid_pair']}")
print("=" * 78)
print("All figures above are measured on the held-out test split, evaluated once. Statements in "
      "Sections 16-20 about feature geometry are exploratory and are labelled as such.")

In [ ]:
# ---- Persist everything -----------------------------------------------------------------------
with open(OUT / "metrics" / "experiment_summary.json", "w") as f:
    json.dump(SUMMARY, f, indent=2, default=str)
with open(OUT / "metrics" / "test_metrics.json", "w") as f:
    json.dump(TEST_METRICS, f, indent=2)
with open(OUT / "metrics" / "config.json", "w") as f:
    json.dump(CONFIG, f, indent=2, default=str)
with open(OUT / "metrics" / "duplicate_report.json", "w") as f:
    json.dump(dup_report, f, indent=2)

index_df.to_csv(OUT / "splits" / "full_file_index_including_unreadable.csv", index=False)
if len(bad):
    bad[["filepath", "class", "error"]].to_csv(OUT / "splits" / "unreadable_files.csv", index=False)

# Final adapter and processor. The best adapter was already written during training; this rewrites
# it from the restored weights so the directory always matches the evaluated model.
model.save_pretrained(OUT / "model" / "final_adapter")
processor.save_pretrained(OUT / "model" / "processor")
with open(OUT / "model" / "adapter_card.json", "w") as f:
    json.dump({"base_model": MODEL_SOURCE, "prompt": CLASSIFICATION_PROMPT,
               "classes": CLASSES, "selected_epoch": int(best["epoch"]),
               "val_macro_f1_at_selection": float(best["macro_f1"]),
               "test_macro_f1": TEST_METRICS["macro_f1"],
               **PARAM_SUMMARY}, f, indent=2)

print("Artifact tree:")
for dirpath, dirnames, filenames in os.walk(OUT):
    depth = len(Path(dirpath).relative_to(OUT).parts)
    print("  " * depth + Path(dirpath).name + "/")
    for fn in sorted(filenames):
        size = os.path.getsize(Path(dirpath) / fn) / 1024
        print("  " * (depth + 1) + f"{fn}  ({size:.1f} KB)")

### What this notebook does and does not establish

**Established, because it is measured on held-out data:** the test accuracy, macro-F1 and weighted-F1
of Qwen3-VL with LoRA on this corpus; the per-class precision, recall and F1; which class pairs the
model actually confuses and in which direction; which test images the model gets confidently wrong.

**Exploratory, and phrased as such throughout:** the t-SNE and UMAP projections, the centroid
similarity matrix, and the relationship between representation-space proximity and observed
confusion. These describe the trained model's internal geometry on a small sample. They are not
evidence about Bangladeshi meme semantics in general, and correlation across ten class pairs is not
a hypothesis test.

**Not addressed here, and worth stating as limitations in the thesis:** label quality is taken as
given; a single seed and one split mean the reported numbers carry sampling variance that is not
quantified (repeat with several seeds and report mean and standard deviation if you need error
bars); any cross-label duplicates reported in Section 05 are label conflicts in the source data that
this pipeline contains but does not resolve.